# Granger-causality analysis

run this notebook in the gc_dvg environment

**Table of Contents**
- [Granger-causality analysis](#Granger-causality-analysis)
- [Data Preparation](#Data-Preparation)
  - [import the data set](#import-the-data-set)
    - [read count data](#read-count-data)
    - [cultivation data](#cultivation-data)
  - [Interpolate the data](#Interpolate-the-data)
  - [Log-normalization](#Log-normalization)
  - [Stationarity](#Stationarity)
    - [plot all stationary time series](#plot-all-stationary-time-series)
    - [plot examples (supplement figure)](#plot-examples-supplement-figure)
- [Granger-causality test](#Granger-causality-test)
  - [Make Granger-causality matrix](#Make-Granger-causality-matrix)
  - [Summarize](#Summarize)
    - [Create summary](#Create-summary)
      - [Summarize DVG labels and time series](#Summarize-DVG-labels-and-time-series)
- [Plot fitted models](#Plot-fitted-models)
  - [Fitting and plotting](#Fitting-and-plotting)
    - [run granger causality prediction and evaluate for all](#run-granger-causality-prediction-and-evaluate-for-all)
      - [all](#all)
      - [corrected](#corrected)
  - [plot single candidates](#plot-single-candidates)
  - [Plot all candidates of a group](#Plot-all-candidates-of-a-group)
- [Swarmplots](#Swarmplots)
  - [all](#all)
  - [corrected](#corrected)
- [group sizes with increasing lag](#group-sizes-with-increasing-lag)
  - [all](#all)
  - [corrected](#corrected)
  - [all vs corrected (supplement figure)](#all-vs-corrected-supplement-figure)
- [SSR with increasing lag (supplement figure)](#SSR-with-increasing-lag-supplement-figure)
  - [corrected](#corrected)
- [Active window analysis for PB2_269_2202 and PB2_129_2176 and PB2_217_2204](#Active-window-analysis-for-PB2_269_2202-and-PB2_129_2176-and-PB2_217_2204)
  - [Poster plots](#Poster-plots)
- [active window analysis general](#active-window-analysis-general)
  - [functions](#functions)
  - [swarmplots](#swarmplots)
  - [histogram](#histogram)
  - [Sankey plots](#Sankey-plots)
  - [experimentally validated candidates (publication panel)](#experimentally-validated-candidates-publication-panel)

First import all necessary packages for Granger-causality analysis & forecasting

In [ ]:
from utils.dip_utils import *
from utils.dip_visuals import *
from utils.granger import (
    calculate_critical_pval_bh,
    correct_p_values_bh_separately,
    create_summary_df,
    granger_labels,
    granger_causation_matrix_fixed_lag,
    make_stationary,
)
from utils.granger_prediction import (
    plot_restricted_only,
    plot_single_candidate,
    run_granger_prediction,
)
from utils.metrics import (
    better_percentage,
    calculate_weighted_mae,
    calculate_weighted_mape,
    cliffs_delta,
    cohens_d,
    dtw_mape,
    dtw_pearson,
    find_local_extrema,
    get_pval_symbol,
    mase,
    nrsme,
    pairwise_mwu_cliffs,
    test_statistical_difference,
)
from utils.plotting import (
    get_corrected_color,
    get_corrected_color_v2,
    get_main_color,
    granger_label_color_map,
    lighten_color,
    make_performance_swarmplot,
    make_performance_swarmplot_with_stats,
)

import matplotlib.pyplot as plt
import numpy as np
import pelz_datasets
import seaborn as sns

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')

# plotting parameters

plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = 'Arial'


In [ ]:
# all outputs will be saved in this folder:
output_prefix = 'data/outputs/plus1_log10_linear_imputation'
! mkdir -p {output_prefix}


# Data Preparation

## import the data set

### read count data

In [ ]:
long_pelz_readcounts_abs = pelz_datasets.long_readcounts(cutoff=15)
long_pelz_readcounts_abs = long_pelz_readcounts_abs.dropna()

measure_ids = long_pelz_readcounts_abs.columns[4:]
long_pelz_readcounts_abs


### cultivation data
Not all values in our data set had sequenced samples so we need to match the measured values with the sequenced values through their time-points

In [ ]:
cultivation_values = pelz_datasets.pelz_cultivation_values()
cultivation_values.drop(columns=['key', 'unit'], inplace=True)
cultivation_values
tp2dpi = cultivation_values.T['dpi'].apply(lambda d : round(d,2)).to_dict()

In [ ]:
read_counts_T = long_pelz_readcounts_abs.copy()
read_counts_T.index = read_counts_T.key
read_counts_T = read_counts_T[long_pelz_readcounts_abs.columns[4:]].T
read_counts_T.index = [round(tp2dpi[idx], 2) for idx in read_counts_T.index]
read_counts_T


In [ ]:
cult_df = cultivation_values.T[['dpi', 'plaque_assay']]
cult_df.index = [round(d,2) for d in cult_df.dpi]
cult_df = cult_df.drop(columns=['dpi'])
cult_df


In [ ]:
ts_df = pd.merge(left=cult_df,
              right=read_counts_T,
              left_index=True,
              right_index=True,
              how='left')
ts_df['plaque_assay'] = pd.to_numeric(ts_df['plaque_assay'], errors='coerce')
ts_df

## Interpolate the data

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax2 = ax.twinx()
ax.set_xlabel('days post infection (dpi)')
ax.plot(ts_df.index,ts_df.drop(columns=['plaque_assay']),
          alpha=0.3, 
          marker='D'
          )
ax.set_ylabel('DVG read counts (abs)')
ax.set_yscale('log')
ax2.plot(ts_df.index, ts_df['plaque_assay'], 
        color='black', 
        marker='o', 
        label='PFU/ml',
        linewidth=3
        )
ax2.set_ylabel('plaque forming units (PFU/ml)')
ax2.set_yscale('log')
ax2.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int)), 1)])
fig.suptitle('Raw data with missing values (log scale)')
plt.show()

list missing time points:

In [ ]:
plaque_assay_missing_timepoints = ts_df[ts_df['plaque_assay'].isna()].index.tolist()
read_counts_missing_timepoints = ts_df[ts_df.drop(columns=['plaque_assay']).isna().any(axis=1)].index.tolist()
print(plaque_assay_missing_timepoints, read_counts_missing_timepoints)

linear interpolation is used to fill in missing time points in the data set. This ensures that the time series data is continuous and can be used for Granger-causality analysis.

In [ ]:

lin_interpolated_ts_df = ts_df.copy()
lin_interpolated_ts_df['plaque_assay'] = lin_interpolated_ts_df['plaque_assay'].interpolate(method='linear')
lin_interpolated_ts_df = lin_interpolated_ts_df.interpolate(method='linear')
lin_interpolated_ts_df.to_csv(f'{output_prefix}/lin_interpolated_ts_df.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax2 = ax.twinx()
ax.set_xlabel('days post infection (dpi)')
ax.plot(lin_interpolated_ts_df.index,lin_interpolated_ts_df.drop(columns=['plaque_assay']),
          alpha=0.3, 
          marker='D'
          )
ax.plot(read_counts_missing_timepoints, lin_interpolated_ts_df.loc[read_counts_missing_timepoints].drop(columns=['plaque_assay']),
          alpha=0.8, 
          marker='X',
          color='tab:red',
          linewidth=0
          )
ax.set_ylabel('DVG read counts (abs)')
ax2.plot(lin_interpolated_ts_df.index, lin_interpolated_ts_df['plaque_assay'], 
        color='black', 
        marker='o', 
        label='PFU/ml',
        linewidth=3
        )
ax2.plot(plaque_assay_missing_timepoints, 
        lin_interpolated_ts_df.loc[plaque_assay_missing_timepoints]['plaque_assay'],
        marker='X',
        color='tab:blue',
        linewidth=0,
        zorder=10
        )
ax2.set_yscale('log')
ax2.set_ylabel('plaque forming units (PFU/ml)')
ax2.set_xticks([range(0, max(lin_interpolated_ts_df.index.astype(int))+1)[i] for i in range(0, max(lin_interpolated_ts_df.index.astype(int)), 1)])
fig.suptitle('Raw data with missing values')
plt.show()


## Log-normalization
To get our data on a better to handle scale (mathematically), we perform log transformation on the NGS-readcounts and the cultivation values: plaque_assay

In [ ]:
log_interpolated_ts_df = lin_interpolated_ts_df.applymap(lambda x: np.log10(x+1))
log_interpolated_ts_df

## Stationarity

Time-series analysis relies on stationarity so that the models and predictions can be statistically relevant and to avoid the detection of spurious relationships.
Thus we need to pre process the imported raw data. I used two methods to obtain stationarity:
1. The iterative method
-> repeatedly form the difference of one value at index i with its preceeding value i-1 in the input array

2. The direct method 
-> directly determine which increment is needed to get a stationary time-series (determine x if np.diff(x) gives us a stationary dataframe) (not used in this notebook)

In [ ]:
stationarity_results = make_stationary(log_interpolated_ts_df, max_diff=10, autolag='AIC', max_lag=None)


In [ ]:
stationary_ts_df = stationarity_results.df_stationary.fillna(0)
stationary_ts_df


### plot all stationary time series

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
ax2 = ax.twinx()
ax.set_xlabel('days post infection (dpi)')
ax.plot(stationary_ts_df.index,stationary_ts_df.drop(columns=['plaque_assay']),
          alpha=0.3, 
          marker='D'
          )
ax.plot(read_counts_missing_timepoints, stationary_ts_df.loc[read_counts_missing_timepoints].drop(columns=['plaque_assay']),
          alpha=0.8, 
          marker='X',
          color='tab:red',
          linewidth=0
          )
ax.set_ylabel('log10(DVG read counts + 1) (abs)')
ax2.plot(stationary_ts_df.index, stationary_ts_df['plaque_assay'], 
        color='black', 
        marker='o', 
        label='PFU/ml',
        linewidth=3
        )
ax2.plot(plaque_assay_missing_timepoints, 
        stationary_ts_df.loc[plaque_assay_missing_timepoints]['plaque_assay'],
        marker='X',
        color='tab:blue',
        linewidth=0,
        zorder=10
        )
ax2.set_yscale('linear')
ax2.set_ylabel('log10(plaque forming units + 1) (PFU/ml)')
ax2.set_xticks([range(0, max(stationary_ts_df.index.astype(int))+1)[i] for i in range(0, max(stationary_ts_df.index.astype(int)), 1)])
fig.suptitle('Raw data with missing values')
plt.show()


In [ ]:
# export
stationary_ts_df.to_csv(f'{output_prefix}/stationary_ts_df.csv')

In [ ]:
stationary_ts_df

### plot examples (supplement figure)

In [ ]:
# =====================================================================
# Supplement figure: stationarity transformation (before -> after)
# Run AFTER the make_stationary step (needs `log_interpolated_ts_df`
# and `stationarity_results` in memory).
# =====================================================================

before_df = log_interpolated_ts_df                       # log10(x+1), interpolated
after_df  = stationarity_results.df_stationary           # stationary (possibly differenced)
diff_lv   = stationarity_results.diff_levels             # per-candidate differencing order
final_p   = stationarity_results.pvals                   # per-candidate final ADF p-value

# --- reproduce the ADF settings used inside make_stationary (for the "before" p-value) ---
def adf_pvalue(series, regression='c', autolag='AIC'):
    x = np.asarray(series.dropna(), dtype=float)
    n = len(x)
    max_lag = int(np.sqrt(max(n, 1)))
    max_lag = max(1, min(max_lag, max(1, n // 2 - 1)))
    with np.errstate(divide='ignore', invalid='ignore'):
        return adfuller(x, autolag=autolag, regression=regression, maxlag=max_lag)[1]

plt.rcParams.update({'font.size': 28})

# --- choose representative examples: one per differencing order, then plaque_assay ---
dvg_cols = [c for c in after_df.columns if c != 'plaque_assay']
examples = []
for lvl in sorted(set(int(diff_lv[c]) for c in dvg_cols if np.isfinite(diff_lv.get(c, np.nan)))):
    pick = next((c for c in dvg_cols if np.isfinite(diff_lv.get(c, np.nan)) and int(diff_lv[c]) == lvl), None)
    if pick is not None:
        examples.append(pick)
    if len(examples) >= 3:
        break
if 'plaque_assay' in after_df.columns:
    examples.append('plaque_assay')

# --- plot: rows = examples, cols = [before, after] ---
n = len(examples)
fig, axes = plt.subplots(n, 2, figsize=(21, 5 * n), squeeze=False)
fig.subplots_adjust(hspace=0.55, wspace=0.25)

for i, col in enumerate(examples):
    b = before_df[col].dropna()
    a = after_df[col].dropna()
    order = int(diff_lv[col]) if np.isfinite(diff_lv.get(col, np.nan)) else None
    p_before = adf_pvalue(before_df[col])
    p_after  = final_p.get(col, np.nan)
    label = 'infectious virus (PFU/mL)' if col == 'plaque_assay' else col

    axL, axR = axes[i, 0], axes[i, 1]
    axL.plot(b.index, b.values, marker='o', color='tab:blue')
    axL.set_ylabel('log10(PFU/mL + 1)' if col == 'plaque_assay' else 'log10(count + 1)')
    axL.set_title(f'{label}\nbefore: ADF p = {p_before:.3f}'
                  + ('  (non-stationary)' if p_before > 0.05 else '  (stationary)'))

    axR.plot(a.index, a.values, marker='D', color='tab:green')
    axR.axhline(0, color='grey', lw=0.8, ls='--')
    ord_txt = 'no differencing' if order == 0 else f'differencing order d = {order}'
    axR.set_ylabel('differenced value' if order else 'log10(count + 1)')
    axR.set_title(f'after: ADF p = {p_after:.3f}\n{ord_txt}')

    for ax in (axL, axR):
        ax.set_xlabel('Time post infection (days)')

fig.suptitle('Stationarity transformation using Augmented-Dickey-Fuller (ADF) test of selected time series (before vs. after)',
             y=1.005)
fig.tight_layout()
fig.savefig(f'{output_prefix}/stationarity_transformation_examples.png',
            dpi=300, bbox_inches='tight')
plt.show()

# Granger-causality test

In [ ]:
# importing the granger causality test from statsmodels
from statsmodels.tsa.stattools import grangercausalitytests

# assigning the string 'ssr_chi2test' to the variable 'test'
test = 'ssr_chi2test'
err_dips = {}

In [ ]:
stationary_ts_df = stationary_ts_df.astype('float64')

## Make Granger-causality matrix

How to read the matrix: column X Granger-causes row Y
i.e. X improves the forecasting performance of Y if included in an OLS model

In [ ]:
import pickle
if not os.path.exists(output_prefix):
  os.makedirs(output_prefix)
  
gc_matrix = {}
test_results = {}
gc_max_lag = {}
errors = {}

corrected_gc_matrix = {}
bh_critical_p_vals = {}

In [ ]:
max_fixed_lag = 4

In [ ]:
for fix_lag in range(1,max_fixed_lag):
    print(f'Calculating granger causality matrix for fixed lag {fix_lag}...')
    gc_matrix[fix_lag], test_results[fix_lag], gc_max_lag[fix_lag], errors[fix_lag] = granger_causation_matrix_fixed_lag(
        stationary_ts_df,
        variables=stationary_ts_df.columns.tolist(),
        cultivation_values=['plaque_assay'],
        fixlag=fix_lag,
        maxlag=fix_lag
    )
    
    corrected_gc_matrix[fix_lag] = correct_p_values_bh_separately(
        gc_matrix[fix_lag],
        cult_columns=1,
        alpha=0.05,
        debug=False
    )
    
    bh_critical_p_vals[fix_lag] = calculate_critical_pval_bh(
        gc_matrix[fix_lag],
        cult_columns=1,
        alpha=0.05,
        debug=False
    )

In [ ]:
print("critical p-values with BH correction:")
bh_critical_p_vals

In [ ]:
import matplotlib.pyplot as plt
import math

# set font size
plt.rcParams.update({'font.size': 8})

# Calculate grid dimensions (e.g., 3 columns)
n_plots = len(gc_matrix)
cols = 6
rows = math.ceil(n_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 2 * rows), constrained_layout=True)
axes = axes.flatten()  # Flatten to iterate easily


for i, fixlag in enumerate(range(1, n_plots + 1)):
    ax = axes[i]
    
    # Plotting the histogram on the specific axis
    # Note: using fixlag instead of the hardcoded 3 from your snippet
    gc_matrix[fixlag].iloc[0].hist(bins=50, ax=ax)
    
    ax.set_title(f'lag={fixlag}')
    ax.set_xlabel('p-value')
    ax.set_ylabel('Frequency')

# Hide any empty subplots if the grid is larger than the number of lags
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Set the main title for the entire figure
plt.suptitle("Histogram of p-values for granger causality DVG->PFU", fontsize=10)
plt.show()

fig_pfu_to_dvg, axes_pfu_to_dvg = plt.subplots(rows, cols, figsize=(15, 2 * rows), constrained_layout=True)
axes_pfu_to_dvg = axes_pfu_to_dvg.flatten()  # Flatten to iterate easily
for i, fixlag in enumerate(range(1, n_plots + 1)):
    ax = axes_pfu_to_dvg[i]
    
    # Plotting the histogram on the specific axis
    # Note: using fixlag instead of the hardcoded 3 from your snippet
    gc_matrix[fixlag].iloc[1:,0].hist(bins=50, ax=ax)
    
    ax.set_title(f'lag={fixlag}')
    ax.set_xlabel('p-value')
    ax.set_ylabel('Frequency')
# Hide any empty subplots if the grid is larger than the number of lags
for j in range(i + 1, len(axes_pfu_to_dvg)):
    axes_pfu_to_dvg[j].axis('off')
# Set the main title for the entire figure
plt.suptitle("Histogram of p-values for granger causality PFU->DVG", fontsize=10)
plt.show()

plt.rcParams.update({'font.size': 20})

## Summarize

### Create summary

In [ ]:
read_counts_df = pd.DataFrame(columns=['key'])
read_counts_df = pd.concat([read_counts_df, stationary_ts_df.T[1:].reset_index().rename(columns={'index': 'key'})], ignore_index=True)


In [ ]:
# Three labellings of the same DVGs:
#   plain      -- raw p-values against a hard 0.05 cut-off
#   corrected  -- BH-adjusted p-values against the same 0.05 cut-off
#   bh         -- raw p-values against the BH critical value per direction
summary_df = {}
summary_df_corrected = {}
summary_df_bh = {}

for fixlag in range(1, max_fixed_lag):
  common = dict(
      gc_test_results=test_results[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      diff_level_dct=stationarity_results.diff_levels.to_dict(),
      readcounts_data=read_counts_df,
      time_series_data=stationary_ts_df,
      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
      cultivation_values=['plaque_assay'],
  )

  summary_df[fixlag] = create_summary_df(
      gc_matrix=gc_matrix[fixlag],
      output_file=f'{output_prefix}/gc_summary_df_lag{fixlag}.csv',
      **common,
  )

  summary_df_corrected[fixlag] = create_summary_df(
      gc_matrix=corrected_gc_matrix[fixlag],
      output_file=f'{output_prefix}/gc_summary_df_corrected_lag{fixlag}.csv',
      **common,
  )

  summary_df_bh[fixlag] = create_summary_df(
      gc_matrix=gc_matrix[fixlag],
      bh_critical_pvals=bh_critical_p_vals[fixlag],
      output_file=f'{output_prefix}/gc_summary_df_bh_lag{fixlag}.csv',
      **common,
  )

In [ ]:
summary_df[1][summary_df[1].key == 'PB2_269_2202']

In [ ]:
summary_df[1].plaque_assay_granger_label.value_counts()

#### Summarize DVG labels and time series
and export to a csv file for further analysis

In [ ]:
stationary_df = stationary_ts_df.T  # keys × timepoints

with_gc_labels = pd.DataFrame(index=stationary_df.index)

key2label = {}
for fixlag in range(1, max_fixed_lag):
    for cultivation_value in ['plaque_assay']:
        for idx, row in summary_df[fixlag].iterrows():
            key = row['key']
            label = row[cultivation_value + '_granger_label']
            if key not in key2label:
                key2label[key] = {}
            key2label[key][f'lag{fixlag}_{cultivation_value}'] = label

            with_gc_labels.loc[key, f'lag{fixlag}_{cultivation_value}_granger_label'] = label

label_cols = [c for c in with_gc_labels.columns if c.endswith('_granger_label')]
with_gc_labels = pd.concat([with_gc_labels, stationary_df], axis=1)

# drop rows if all label cols are NaN
with_gc_labels = with_gc_labels.dropna(subset=label_cols, how='all')

# drop time-series cols if all rows are NaN
ts_cols = [c for c in with_gc_labels.columns if c not in label_cols]
keep = [c for c in ts_cols if with_gc_labels[c].notna().any()]
with_gc_labels = with_gc_labels[label_cols + keep]

with_gc_labels.to_csv(f'{output_prefix}/dvg_time_series_with_gc_labels.csv')
with_gc_labels

# Plot fitted models

## Fitting and plotting

### run granger causality prediction and evaluate for all

#### all

In [ ]:
summary_df[1].sort_values(by='plaque_assay_ssr_chi2test').value_counts(subset=['plaque_assay_granger_label'])

In [ ]:
plt.rcParams.update({'font.size': 11, 'font.family': 'Arial'})
gc_prediction_summary = {}
ref_ols_performance_dct = {}
for fixlag in range(1, max_fixed_lag):
  print(f'Running granger prediction analysis for lag {fixlag}...')
  ref_ols_performance_dct[fixlag] = {}
  gc_prediction_summary[fixlag], ref_ols_performance_dct[fixlag] = run_granger_prediction(
      summary_df=summary_df[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      gc_test_results=test_results[fixlag],
      plot_dips=[ 'PB2_269_2202'#,'PB2_129_2176','PB2_217_2204,
                #,'PA_1989_2027','PA_172_1938', 'PA_124_1974', 'PA_142_1951'
                ],
      log_long_data=stationary_ts_df,
      restricted_pred_index=summary_df[fixlag].iloc[0].name,
      include_ssr=True,
      figsize=(4,2),
      ylabel='log10(PFU/mL+1)'
  )

#### corrected

In [ ]:
plt.rcParams.update({'font.size': 11})
gc_prediction_summary_corrected = {}
ref_ols_performance_corrected_dct = {}
for fixlag in range(1, max_fixed_lag):
  print(f'Running granger prediction analysis with p-value correction for lag {fixlag}...')
  gc_prediction_summary_corrected[fixlag], ref_ols_performance_corrected_dct[fixlag] = run_granger_prediction(
      summary_df=summary_df_corrected[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      gc_test_results=test_results[fixlag],
      log_long_data=stationary_ts_df,
      restricted_pred_index=summary_df_corrected[fixlag].iloc[0].name,
      include_ssr=True,
      plot_dips=[#"PB2_129_2176", 
                #"PB2_269_2202", 
                "PB2_217_2204"],
      figsize=(4,2), ylabel='log10(PFU/ml+1)'
  )

## plot single candidates

In [ ]:
plt.rcParams.update({'font.size': 18})
gc_prediction_summary_sorted = gc_prediction_summary[1].sort_values(by='ssr')

for dvg in ['PB1_103_2173', 'PB1_126_2139']:
  print(f"Plotting {dvg}...")
  plot_single_candidate(cultivation_value_label ='plaque_assay',
                          dpis = stationary_ts_df.index.values,
                          row = summary_df_corrected[1][summary_df_corrected[1].key == dvg].iloc[0],
                          figsize = (5, 4),
                          xlabel = 'Time post infection (days)',
                          ylabel = 'log10(PFU/mL+1)',
                          yscale = 'linear',
                          opt_lag = 1,
                          gc_prediction_summary_df = gc_prediction_summary[1],
                          gc_test_results= test_results[1],
                          log_long_data = stationary_ts_df,
                          style = 'detailed')

## Plot all candidates of a group

In [ ]:
import seaborn as sns
def plot_predictions_by_label(
  gc_prediction_summary_df,
  gc_test_results,
  cultivation_value_label='plaque_assay',
  log_long_data=stationary_ts_df,
  label_order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
  figsize=(30, 4),
  output_prefix='',
  lag=1
):
  first_row = gc_prediction_summary_df.iloc[0]
  opt_lag = first_row['optimal_lag']
  restricted_pred = gc_test_results[(cultivation_value_label, first_row['key'])][opt_lag][1][0].predict()
  actual_value = log_long_data[cultivation_value_label]
  dpis = log_long_data.index.values
  
  fig, axs = plt.subplots(figsize=figsize, nrows=1, ncols=len(label_order), sharey=True)
  axs = axs.flatten()
  
  for label_idx, label in enumerate(label_order):
    label_df = gc_prediction_summary_df[gc_prediction_summary_df['granger_label'] == label]
    
    if label_df.empty:
      axs[label_idx].text(0.5, 0.5, 'No data', ha='center', va='center', 
                          transform=axs[label_idx].transAxes)
      axs[label_idx].set_title(f"{label} (0)")
      continue
    
    ax = axs[label_idx]
    
    # Aggregate predictions by DPI
    pred_by_dpi = {}
    for idx, row in label_df.iterrows():
      pred_dpis, pred_values = row['pred_data']
      for dpi, val in zip(pred_dpis, pred_values):
        pred_by_dpi.setdefault(dpi, []).append(val)
    
    pred_dpis_sorted = sorted(pred_by_dpi.keys())
    pred_means = [np.mean(pred_by_dpi[dpi]) for dpi in pred_dpis_sorted]
    pred_stds = [np.std(pred_by_dpi[dpi]) for dpi in pred_dpis_sorted]
    
    sns.lineplot(x=dpis, y=actual_value, marker='s', color='black', 
                label='Actual', linewidth=2, ax=ax)
    sns.lineplot(x=dpis[opt_lag:], y=restricted_pred, marker='D', color='tab:orange', 
                label='Restricted Model', linewidth=2, ax=ax)
    sns.lineplot(x=pred_dpis_sorted, y=pred_means, marker='o', 
                color=granger_label_color_map.get(label, 'gray'),
                label='Full model', linewidth=2, ax=ax)
    
    ax.fill_between(pred_dpis_sorted, np.array(pred_means) - np.array(pred_stds),
                    np.array(pred_means) + np.array(pred_stds),
                    color=granger_label_color_map.get(label, 'gray'), alpha=0.2)
    
    ax.set_title(f"{label} ({len(label_df)})")
    ax.set_ylabel('log10(PFU/mL)' if label_idx == 0 else '')
    ax.set_xlabel('DPI')
    ax.set_ylim(-1, 10)
    
    # Reorder legend
    handles, labels = ax.get_legend_handles_labels()
    order = [labels.index(name) for name in ['Restricted Model', 'Full model', 'Actual']]
    ax.legend([handles[i] for i in order], [labels[i] for i in order],
              loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
  
  fig.suptitle(f'Predictions by Granger-causality label (Lag={lag})', y=1.01)
  plt.tight_layout()
  
  if output_prefix:
    fig.savefig(f"{output_prefix}/forecast_plots/predictions_by_label_lag{lag}.png", 
                dpi=300, bbox_inches='tight')
  plt.show()

In [ ]:
for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag}...")
  plot_predictions_by_label(
    gc_prediction_summary_df=gc_prediction_summary[lag],
    gc_test_results=test_results[lag],
    cultivation_value_label='plaque_assay',
    figsize=(20, 5),
    log_long_data=stationary_ts_df,
    label_order=['causing', 'bi-directional', 'caused', 'non-related'],
    lag=lag
  )
  
for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag} with p-value correction...")
  plot_predictions_by_label(
      gc_prediction_summary_df=gc_prediction_summary_corrected[lag],
      gc_test_results=test_results[lag],
      cultivation_value_label='plaque_assay',
      figsize=(20, 5),
      log_long_data=stationary_ts_df,
      label_order=['causing', 'bi-directional', 'caused', 'non-related'],
      lag=lag
  )

# Swarmplots

## all 

In [ ]:
plt.rcParams.update({'font.size': 28})
! mkdir -p {output_prefix}/plots


In [ ]:
gc_prediction_summary[1][gc_prediction_summary[1].key == 'PB2_269_2202']

In [ ]:
plt.rcParams.update({'font.size': 28})
for fixlag in range(1, max_fixed_lag):
  print(f'Creating performance swarmplot for lag {fixlag}...')
  fig, ax = make_performance_swarmplot(
      dip_forecast_summary=gc_prediction_summary[fixlag],
      ref_performance_dct=ref_ols_performance_dct[fixlag],
      performance_metric='ssr',
      forecasted_value = 'plaque_assay',
      title = f'Granger-related DVGs predictive power over PFU (lag={fixlag})',
      ylabel='SSR',
      dot_color_ref = 'group_norm_ssr_chi2test_pval',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=ref_ols_performance_dct[fixlag]['SSR']*2,
      ymin=-10,
      p_val_pos=ref_ols_performance_dct[fixlag]['SSR']*1.5,
      figsize=(18,7),
      dotsize=8,
      linewidth=5
  )
  fig.savefig(f'{output_prefix}/plots/ssr_swarmplot_lag{fixlag}.png')

## corrected

In [ ]:
plt.rcParams.update({'font.size': 28})
for fix_lag in range(1,max_fixed_lag):
  fig, ax = make_performance_swarmplot(gc_prediction_summary_corrected[fix_lag],
                              ref_ols_performance_corrected_dct[fix_lag],
                              performance_metric='ssr',
                              forecasted_value = 'plaque_assay',
                              title = f'Granger-related DI vRNAs predictive power over PFU with p-value correction (lag={fix_lag})',
                              ylabel='SSR',
                              dot_color_ref = 'group_norm_ssr_chi2test_pval',
                              inverted_colors=True,
                              upper_border=-0.05,
                              ymax=ref_ols_performance_corrected_dct[fix_lag]['SSR']*1.5,
                              ymin=-10,
                              p_val_pos=ref_ols_performance_corrected_dct[fix_lag]['SSR']*1.2,
                              figsize=(18,8),
                              dotsize=8,
                              linewidth=5
                              )
  fig.savefig(f'{output_prefix}/plots/ssr_swarmplot_corrected_lag{fix_lag}.png')
  fig.show()

In [ ]:
ref_ols_performance_df = pd.DataFrame(ref_ols_performance_dct).T.reset_index().rename(columns={'index': 'lag'})
for lag in range(1, max_fixed_lag):
    gc_prediction_summary[lag].to_csv(f'{output_prefix}/gc_prediction_summary_lag{lag}.csv', index=False)
    gc_prediction_summary_corrected[lag].to_csv(f'{output_prefix}/gc_prediction_summary_corrected_lag{lag}.csv', index=False)

# group sizes with increasing lag

## all

In [ ]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs across different fixed lags')
# Add legend next to the plot
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

## corrected

In [ ]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs across different fixed lags (with p-value correction)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

## all vs corrected (supplement figure)

In [ ]:
plt.rcParams.update({'font.size': 11})
fig, ax = plt.subplots(figsize=(6, 4), dpi=800)

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []
  group_sizes_corrected = [] 
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
    
    subset_corrected = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    group_sizes_corrected.append(len(subset_corrected))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color, linestyle='solid')
  ax.plot(range(1, max_fixed_lag), group_sizes_corrected, marker='D', label=f'{label} (corrected)', color=color, linestyle='dashed', markeredgewidth=1, markeredgecolor=color, markerfacecolor='white')
ax.set_xlabel('Fixed lag for Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs\nwith increasing fixed lags')
ax.set_yscale('log')
# Add legend next to the plot
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

# SSR with increasing lag (supplement figure)

In [ ]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3), dpi=800)
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []
for fixlag in range(1, max_fixed_lag):
    mean_ssr_full_model.append(gc_prediction_summary[fixlag]['ssr'].median())
    ax.scatter(fixlag,
              gc_prediction_summary[fixlag]['ssr'].median(),
              color='tab:green',
              label='full model' if fixlag==1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 5))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []


for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  mean_ssr_full_model = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    mean_ssr_full_model.append(subset['ssr'].median())
    ax.scatter(fixlag,
              subset['ssr'].median(),
              color=color,
              label=label if fixlag==1 else "",
              linewidth=2)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
plt.show()

## corrected

In [ ]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 5))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_corrected_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []


for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  mean_ssr_full_model = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    mean_ssr_full_model.append(subset['ssr'].median())
    ax.scatter(fixlag,
              subset['ssr'].median(),
              color=color,
              label=label if fixlag==1 else "",
              linewidth=2)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()

# Active window analysis for PB2_269_2202 and PB2_129_2176 and PB2_217_2204

In [ ]:
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

def classify_from_pvals(p_causing, p_caused, alpha=0.05):
    c, d = p_causing <= alpha, p_caused <= alpha
    if c and d: return 'bi-directional'
    if c:       return 'causing'
    if d:       return 'caused'
    return 'non-related'
  
full_actual_values = log_interpolated_ts_df['plaque_assay'].astype(float)
actual_dpis = log_interpolated_ts_df.index.values

plt.rcParams.update({'font.size': 18, 'font.family': 'Arial'})

# ΔPFU/mL per candidate (value string, box color)
delta_pfu_map = {
    'PB2_217_2204': ('2.4e4', "#c6f8ff"),   # cyan
    'PB2_129_2176': ('5.7e3', '#ffe599'),   # yellow
    'PB2_269_2202': ('2.8e5', '#ea9edb'),   # pink
}

def analyze_and_plot_candidate(key, window_df, figsize=(5, 4), title_prefix=''):
    """PFU (plaque_assay) is kept raw/undifferenced; only the DVG read count is
    stationarized. Granger is run, and predictions are plotted, on raw log10(PFU)."""
    opt_lag = 1

    # ---- stationarize ONLY the DVG column; keep PFU raw ----
    dvg_res = make_stationary(window_df[[key]], max_diff=10, autolag='AIC', max_lag=None)
    d_dvg = int(dvg_res.diff_levels[key]) if np.isfinite(dvg_res.diff_levels[key]) else 0
    if key not in dvg_res.df_stationary.columns:
        raise ValueError(f"{key}: DVG series was dropped during stationarization")
    dvg_stat = dvg_res.df_stationary[key].fillna(0)

    pfu_raw = window_df['plaque_assay'].astype(float)

    # align on shared time points (differencing the DVG drops leading rows)
    gc_df = pd.concat([pfu_raw.rename('plaque_assay'), dvg_stat.rename(key)], axis=1).dropna()
    gc_df = gc_df.astype('float64')
    if len(gc_df) < opt_lag + 2:
        raise ValueError(f"{key}: too few aligned points after DVG differencing (n={len(gc_df)})")

    # causing = DVG -> PFU ; caused = PFU -> DVG  (statsmodels: 2nd col causes 1st)
    causing = grangercausalitytests(gc_df[['plaque_assay', key]], maxlag=1, verbose=False)
    caused  = grangercausalitytests(gc_df[[key, 'plaque_assay']], maxlag=1, verbose=False)

    p_causing = causing[1][0]['ssr_chi2test'][1]
    p_caused  = caused[1][0]['ssr_chi2test'][1]

    # predictions of raw PFU (target is undifferenced), plus SSRs on that same scale
    full_pred  = causing[1][1][1].predict()
    restr_pred = causing[1][1][0].predict()
    SSR_full   = causing[1][1][1].ssr
    SSR_restr  = causing[1][1][0].ssr

    label = classify_from_pvals(p_causing, p_caused)
    full_color = granger_label_color_map[label]

    dpis = gc_df.index.values                    # time axis after alignment
    actual = full_actual_values              # raw log10(PFU) on the aligned grid

    print(f"{key}: label={label}  p_causing={p_causing:.4f}  p_caused={p_caused:.4f}  "
          f"SSR_restr={SSR_restr:.2f}  SSR_full={SSR_full:.2f}  diff(DVG)={d_dvg}  (n={len(gc_df)})")

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(actual_dpis, actual.values, label='Data', marker='D', color='black')
    ax.plot(dpis[opt_lag:], restr_pred, label='Restricted\nmodel',
            marker='o', markersize=7, color='#e09312',
            markeredgecolor='#e09312', markerfacecolor="#e1bf83",
            markeredgewidth=2, linewidth=4, linestyle=(0, (1, 1)))
    ax.plot(dpis[opt_lag:], full_pred, label='Full model',
            marker='o', markersize=7, color=full_color,
            markeredgecolor=full_color, markerfacecolor=lighten_color(full_color, 0.5),
            markeredgewidth=2, linewidth=4, linestyle='dashed')
    ax.set_title(f'{title_prefix}{key} (lag={opt_lag})')
    ax.set_xlabel('Time post infection (days)')
    ax.set_ylabel('log10(PFU/mL+1)')                 # always raw -> label is correct

    # --- x-axis: major ticks 0-22 step 2, minor ticks between ---
    ax.xaxis.set_major_locator(MultipleLocator(2))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    # --- y-axis: end at 10 ---
    ax.set_ylim(0, 10)

    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.6), ncol=1)
    ax.set_yscale('linear')
    
    fig.text(1.1, -0.05, f'SSR$_{{full}}$ = {SSR_full:.2f}', transform=ax.transAxes,
             verticalalignment='top',
             bbox=dict(boxstyle='square', facecolor=full_color, alpha=0.5))
    fig.text(1.1, 0.1, f'SSR$_{{restricted}}$ = {SSR_restr:.2f}', transform=ax.transAxes,
             verticalalignment='top',
             bbox=dict(boxstyle='square', facecolor='#e09312', alpha=0.5))
    d_val, d_color = delta_pfu_map[key]
    fig.text(1.1, 0.25, rf'$\Delta$PFU/mL = {d_val}', transform=ax.transAxes,
             verticalalignment='top',
             bbox=dict(boxstyle='square', facecolor=d_color, alpha=1))
    fig.text(1.1, 1.01, f'Granger-{label}', transform=ax.transAxes,
             verticalalignment='top',
             bbox=dict(boxstyle='square', facecolor=full_color, alpha=0.5))
    plt.show()
    return {'key': key, 'label': label, 'p_causing': p_causing, 'p_caused': p_caused,
            'SSR_full': SSR_full, 'SSR_restricted': SSR_restr, 'diff_dvg': d_dvg, 'n': len(gc_df)}


## Poster plots

In [ ]:
# ---- build windows ----
plt.rcParams.update({'font.size': 18})
first_occ_denovo = ts_df['PB2_269_2202'].gt(15).idxmax()                 # 13.5
denovo_win = log_interpolated_ts_df.loc[first_occ_denovo:, ['plaque_assay', 'PB2_269_2202']].copy()

last_occ_loss = ts_df['PB2_129_2176'].gt(15).iloc[::-1].idxmax()         # 5.48
_pos = ts_df.index.get_loc(last_occ_loss)
loss_end = ts_df.index[min(_pos + 1, len(ts_df) - 1)]                    # 5.99
loss_win = log_interpolated_ts_df.loc[:loss_end, ['plaque_assay', 'PB2_129_2176']].copy()

gain_win = log_interpolated_ts_df.loc[:, ['plaque_assay', 'PB2_217_2204']].copy()   # full 0-21 dpi

print(f"de novo window: {denovo_win.index.min()}–{denovo_win.index.max()} dpi ({len(denovo_win)} pts)")
print(f"loss window:    {loss_win.index.min()}–{loss_win.index.max()} dpi ({len(loss_win)} pts)")
print(f"gain window:    {gain_win.index.min()}–{gain_win.index.max()} dpi ({len(gain_win)} pts)")

results = []
results.append(analyze_and_plot_candidate('PB2_269_2202', denovo_win))
results.append(analyze_and_plot_candidate('PB2_129_2176', loss_win))
results.append(analyze_and_plot_candidate('PB2_217_2204', gain_win))
pd.DataFrame(results).set_index('key')

In [ ]:
pd.DataFrame(ref_ols_performance_dct).T.reset_index().rename(columns={'index': 'lag'}).to_csv(f'{output_prefix}/ref_ols_performance_dct.csv', index=False)

# active window analysis general

## functions

In [ ]:
# ============================================================
# ACTIVE WINDOW ANALYSIS
# Step 1: derive analysis parameters and assign enrichment types
#         (gain / loss / de novo gain) from the raw read-count trajectories.
# ============================================================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from scipy.signal import find_peaks
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.stats.multitest import multipletests

READ_CUTOFF = 15
OPT_LAG = 1
RESTRICTED_COLOR = '#e09312'

# MAX_POINTS: a candidate only enters the active-window BH family if its window
# excludes at least one full oscillation of the infectious virus concentration.
FULL_LEN = len(log_interpolated_ts_df)
_pfu = log_interpolated_ts_df['plaque_assay'].astype(float)
_peaks, _ = find_peaks(_pfu.values, prominence=1.0)          # prominence in log10 units
PFU_PERIOD = int(np.round(np.mean(np.diff(_peaks))))         # mean peak-to-peak spacing
MIN_POINTS = PFU_PERIOD
MAX_POINTS = FULL_LEN #PFU_PERIOD

print(f"PFU peaks at dpi: {list(np.round(_pfu.index.values[_peaks], 2))}")
print(f"oscillation period: {PFU_PERIOD} sampling points")
print(f"full series: {FULL_LEN} points  ->  MIN_POINTS = {MIN_POINTS}, MAX_POINTS = {MAX_POINTS}\n")

# ---- enrichment types ----
rc = read_counts_T.copy()                    # index = dpi, columns = DVG keys
dvg_keys = list(rc.columns)

above_cutoff = rc >= READ_CUTOFF
ever_present = above_cutoff.any(axis=0)
start_rc, end_rc = rc.iloc[0], rc.iloc[-1]

def enrichment_type(key):
    if not ever_present[key]:
        return 'absent'
    start_present = rc.iloc[0][key]
    if (not start_present) and end_rc[key] > start_rc[key]:
        return 'de novo gain'
    if end_rc[key] == 0 and start_rc[key] == 0:
        return 'de novo loss'
    if end_rc[key] >= start_rc[key]:
        return 'gain'
    if end_rc[key] < start_rc[key]:
        return 'loss'
    return 'unchanged'

enrichment = pd.Series({k: enrichment_type(k) for k in dvg_keys}, name='enrichment_type')
print(enrichment.value_counts())

In [ ]:
# ------------------------------------------------------------
# Step 2: per-candidate active window on the log-interpolated series.
# ------------------------------------------------------------
def active_window_df(key, etype=None, threshold=0):
    """
    Active window = the longest contiguous run of values strictly above `threshold`
    in the raw read counts (ts_df), applied uniformly regardless of enrichment type.
    threshold=0 -> non-zero; set threshold=READ_CUTOFF to match the >15 presence rule.
    Returns the corresponding slice of log_interpolated_ts_df, or None if never present.
    """
    present = lin_interpolated_ts_df[key].to_numpy() > threshold
    if not present.any():
        return None

    # find the longest contiguous True run
    best_len = best_start = 0
    cur_start = None
    for i, p in enumerate(present):
        if p and cur_start is None:
            cur_start = i
        elif not p and cur_start is not None:
            if i - cur_start > best_len:
                best_len, best_start = i - cur_start, cur_start
            cur_start = None
    if cur_start is not None and len(present) - cur_start > best_len:
        best_len, best_start = len(present) - cur_start, cur_start

    start_lbl = ts_df.index[best_start]
    end_lbl   = ts_df.index[best_start + best_len - 1]
    window = log_interpolated_ts_df.loc[start_lbl:end_lbl]
    return window[['plaque_assay', key]].copy()

In [ ]:
# ------------------------------------------------------------
# Step 3: for each candidate, fit restricted + full OLS on its active
#         window (PFU raw, DVG stationarized) and record RAW p-values.
#         Classification happens AFTER BH-correction (Step 4).
# MAE = median absolute error. SSR is not used: windows differ in length
#       and SSR scales with the number of points.
# ------------------------------------------------------------

def active_window_performance(key, window_df, opt_lag=OPT_LAG):
    dvg_res = make_stationary(window_df[[key]], max_diff=10, autolag='AIC', max_lag=None)
    if key not in dvg_res.df_stationary.columns:
        raise ValueError('dvg_dropped')
    dvg_stat = dvg_res.df_stationary[key].fillna(0)
    pfu_raw  = window_df['plaque_assay'].astype(float)

    gc_df = pd.concat([pfu_raw.rename('plaque_assay'), dvg_stat.rename(key)], axis=1).dropna()
    gc_df = gc_df.astype('float64')
    if len(gc_df) < opt_lag + 2:
        raise ValueError('too_few_points')

    causing = grangercausalitytests(gc_df[['plaque_assay', key]], maxlag=1, verbose=False)
    caused  = grangercausalitytests(gc_df[[key, 'plaque_assay']], maxlag=1, verbose=False)

    full_pred  = causing[1][1][1].predict()
    restr_pred = causing[1][1][0].predict()
    actual = gc_df['plaque_assay'].iloc[opt_lag:].values

    ssr_full  = float(causing[1][1][1].ssr)
    ssr_restr = float(causing[1][1][0].ssr)

    return {'key': key,
            'p_causing': causing[1][0]['ssr_chi2test'][1],
            'p_caused':  caused[1][0]['ssr_chi2test'][1],
            'mae_full':       float(np.median(np.abs(actual - full_pred))),
            'mae_restricted': float(np.median(np.abs(actual - restr_pred))),
            'ssr_full': ssr_full,
            'ssr_restricted': ssr_restr,
            'relative_ssr_full': ssr_full / ssr_restr if ssr_restr > 0 else np.nan,
            'relative_ssr_restricted': 1.0,          # baseline by definition
            'enrichment_type': enrichment[key],
            'window_len': len(window_df),
            'full_len': len(log_interpolated_ts_df),
            'n_fitted': len(gc_df)}

perf_records, perf_errors = [], {}
for key in dvg_keys:
    win = active_window_df(key, enrichment[key])
    if win is None or len(win) < MIN_POINTS:
        perf_errors[key] = 'window_too_short_or_absent'
        continue
    try:
        perf_records.append(active_window_performance(key, win))
    except Exception as e:
        perf_errors[key] = str(e)[:40]

perf_df = pd.DataFrame(perf_records).set_index('key')
print(f"evaluated: {len(perf_df)}   skipped: {len(perf_errors)}")
print(Counter(perf_errors.values()).most_common())

In [ ]:
# ------------------------------------------------------------
# Step 4: BH correction.
#
# The BH family is restricted to candidates whose active window excludes
# at least one full PFU oscillation (window_len <= MAX_POINTS).
#
# Rationale: BH is a step-up procedure, so a candidate's adjusted p-value
# depends on the ranks of all other p-values in the family. Candidates whose
# active window is (nearly) the full time series contribute tests that are
# (nearly) identical to those already performed and corrected in the
# full-time-series analysis; including them would both duplicate those tests
# and distort the correction for the candidates that are genuinely restricted.
#
# The criterion is structural (window length), not p-value based.
# ------------------------------------------------------------
perf_df['restricted_window'] = perf_df['window_len'] <= MAX_POINTS

print(f"BH family (active window <= {MAX_POINTS} of {FULL_LEN} points): "
      f"{int(perf_df['restricted_window'].sum())} / {len(perf_df)}")
print(perf_df.groupby('enrichment_type')['restricted_window'].sum())

def bh(pvals):
    p = pvals.to_numpy(dtype=float)
    mask = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    if mask.any():
        out[mask] = multipletests(p[mask], alpha=0.05, method='fdr_bh')[1]
    return pd.Series(out, index=pvals.index)

# --- BH within the restricted-window family only ---
fam = perf_df[perf_df['restricted_window']]
perf_df['p_causing_bh'] = np.nan
perf_df['p_caused_bh']  = np.nan
perf_df.loc[fam.index, 'p_causing_bh'] = bh(fam['p_causing'])
perf_df.loc[fam.index, 'p_caused_bh']  = bh(fam['p_caused'])

# Candidates outside the family keep their full-time-series (BH-corrected) labels.
full_series_labels = summary_df_bh[1].set_index('key')['plaque_assay_granger_label']

labels = {}
for k, r in perf_df.iterrows():
    if r['restricted_window']:
        labels[k] = classify_from_pvals(r['p_causing_bh'], r['p_caused_bh'])
    else:
        labels[k] = full_series_labels.get(k, 'non-related')
perf_df['granger_label'] = pd.Series(labels)

# uncorrected labels (all candidates, raw p) -- for the comparison Sankey
perf_df['granger_label_raw'] = [classify_from_pvals(a, b)
                                for a, b in zip(perf_df['p_causing'], perf_df['p_caused'])]

print("\nactive-window labels (BH within restricted-window family):")
print(perf_df['granger_label'].value_counts())
print("\nuncorrected:")
print(perf_df['granger_label_raw'].value_counts())

# --- p-value distribution within the family: is there signal? ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(fam['p_causing'].dropna(), bins=25, color='tab:blue', edgecolor='white')
axes[0].set_title(f'raw p (causing), BH family (n={len(fam)})'); axes[0].set_xlabel('p-value')
axes[1].hist(fam['p_caused'].dropna(), bins=25, color='tab:red', edgecolor='white')
axes[1].set_title(f'raw p (caused), BH family (n={len(fam)})'); axes[1].set_xlabel('p-value')
fig.tight_layout(); plt.show()

# --- sensitivity of the genome-wide result to the MAX_POINTS cutoff ---
rows = []
for cutoff in range(MIN_POINTS, FULL_LEN + 1, 2):
    f = perf_df[perf_df['window_len'] <= cutoff]
    if len(f) < 10:
        continue
    pc = multipletests(f['p_causing'].dropna(), alpha=0.05, method='fdr_bh')[1]
    pd_ = multipletests(f['p_caused'].dropna(),  alpha=0.05, method='fdr_bh')[1]
    rows.append({'max_points': cutoff, 'family_size': len(f),
                 'n_causing': int((pc <= 0.05).sum()),
                 'n_caused':  int((pd_ <= 0.05).sum())})
sens = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sens['max_points'], sens['n_causing'], marker='o', label='Granger-causing')
ax.plot(sens['max_points'], sens['n_caused'],  marker='o', label='Granger-caused')
ax.axvline(MAX_POINTS, color='black', linestyle='--',
           label=f'chosen cutoff ({MAX_POINTS})')
ax.set_xlabel('MAX_POINTS (maximum active window length in the BH family)')
ax.set_ylabel('BH-significant candidates')
ax.legend(); fig.tight_layout(); plt.show()
print(sens.to_string(index=False))

# --- the three experimentally validated candidates (pre-specified, uncorrected) ---
check = [k for k in ['PB2_269_2202', 'PB2_129_2176', 'PB2_217_2204'] if k in perf_df.index]
print(perf_df.loc[check, ['enrichment_type', 'window_len', 'restricted_window',
                          'p_causing', 'p_causing_bh',
                          'granger_label_raw', 'granger_label']])

In [ ]:
perf_df[perf_df.index.isin(['PB2_269_2202', 'PB2_129_2176', 'PB2_217_2204'])]

## swarmplots

In [ ]:
# ------------------------------------------------------------
# Genome-wide active-window results: swarmplots
#   restricted | causing | bi-directional | caused | non-related
# Dot color = active window length (single continuous colormap)
# Metrics:
#   'mae'          median absolute error (per-point, comparable)
#   'ssr'          raw sum of squared residuals (scales with window length!)
#   'relative_ssr' SSR_full / SSR_restricted; restricted group omitted (== 1.0)
# ------------------------------------------------------------
import matplotlib as mpl

LABEL_ORDER = ['causing', 'bi-directional', 'caused', 'non-related']
plt.rcParams.update({'font.size': 12})

METRIC_INFO = {
    # metric: (ylabel, reference hline, include restricted swarm?)
    'mae':          ('Median Absolute Error (MAE)',      None, True),
    'ssr':          ('Sum of Squared Residuals (SSR)',   None, True),
    'relative_ssr': ('Relative SSR (full / restricted)', 1.0,  False),
}

def build_swarm_df(perf_df, metric='mae', label_col='granger_label', include_restricted=True):
    full_col, restr_col = f'{metric}_full', f'{metric}_restricted'
    cols = ['key', 'window_len']
    full_long = (perf_df.reset_index()[cols + [label_col, full_col]]
                        .rename(columns={label_col: 'granger_label', full_col: 'value'}))
    if not include_restricted:
        out = full_long[cols + ['granger_label', 'value']].copy()
    else:
        restr_long = (perf_df.reset_index()[cols + [restr_col]]
                             .rename(columns={restr_col: 'value'}))
        restr_long['granger_label'] = 'restricted'
        out = pd.concat([restr_long[cols + ['granger_label', 'value']],
                         full_long[cols + ['granger_label', 'value']]], ignore_index=True)
    return out

def make_active_window_swarmplot(perf_df, metric='mae', label_col='granger_label',
                                 cmap='viridis',
                                 title=None, figsize=(8, 4), dotsize=3, markersize=18,
                                 ymin=0, ymax=None, p_val_pos=0):
    ylabel, hline, include_restricted = METRIC_INFO[metric]
    swarm_df = build_swarm_df(perf_df, metric, label_col, include_restricted)
    order = (['restricted'] + LABEL_ORDER) if include_restricted else LABEL_ORDER

    vmin = int(swarm_df['window_len'].min())
    vmax = int(swarm_df['window_len'].max())
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=figsize, dpi=800)
    sns.swarmplot(x='granger_label', y='value', data=swarm_df, order=order,
                  hue='window_len', palette=cmap, hue_norm=norm,
                  size=5, ax=ax, legend=False, linewidth=0.2)

    ax.set_title(title or 'Active-window performance by Granger-causality label\n', pad=20)
    ax.set_xlabel('Granger-causality label', labelpad=10)
    ax.set_ylabel(ylabel)
    if ymax is None:
        ymax = swarm_df['value'].quantile(0.99) * 1.15
    ax.set_ylim(ymin, ymax)

    for x, ticklabel in zip(ax.get_xticks(), ax.get_xticklabels()):
        lab = ticklabel.get_text()
        sub = swarm_df[swarm_df.granger_label == lab]['value']
        if len(sub):
            ax.plot(x, sub.median(), marker='+', color='black',
                    markersize=markersize, zorder=11)
            print(metric, 'sub.median', lab, sub.median())
        ax.text(x, p_val_pos, f'n={len(sub)}', ha='center', va='bottom',
                color='black', zorder=10)

    xmin, xmax = ax.get_xlim()
    if hline is not None:
        ax.hlines(y=hline, xmin=xmin, xmax=xmax, color=RESTRICTED_COLOR,
                  linewidth=3, linestyle='--', zorder=9,
                  label='restricted model (no improvement)')
    else:
        restr_median = swarm_df[swarm_df.granger_label == 'restricted']['value'].median()
        ax.hlines(y=restr_median, xmin=xmin, xmax=xmax, color=RESTRICTED_COLOR,
                  linewidth=3, linestyle='--', zorder=9,
                  label='median restricted model performance')
    ax.legend(loc='lower left', bbox_to_anchor=(0.02, 1.01), frameon=False)
    ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)

    # ---- colorbar: ticks at the actual min and max window length ----
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_ticks([vmin, vmax])
    cbar.set_ticklabels([str(vmin), str(vmax)])
    cbar.set_label('Active window length (time points)', labelpad=10)

    fig.tight_layout()
    return fig, ax

# ---- genome-wide swarmplots ----
for metric in [#'mae', 
              #'ssr', 
              'relative_ssr']:
    fig, ax = make_active_window_swarmplot(perf_df, metric=metric)
    plt.show()
    fig.savefig(f'{output_prefix}/plots/active_window_{metric}_swarmplot.pdf', bbox_inches='tight')

In [ ]:
# ------------------------------------------------------------
# Does the separation between Granger groups shrink for short
# active windows?  (i.e. is the labeling less powerful with fewer
# time points, or is the group separation stable?)
#
# Compares causing vs non-related WITHIN window-length strata,
# using relative SSR (full/restricted) and Cliff's delta.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

d = perf_df.copy()
d['rel_ssr'] = d['relative_ssr_full']

# --- stratify by active window length ---
# quartiles, plus an explicit "full series" stratum since it dominates
d['wstratum'] = pd.qcut(d['window_len'], 4, duplicates='drop')

rows = []
for stratum, sub in d.groupby('wstratum', observed=True):
    causing = sub.loc[sub['granger_label'].isin(['causing', 'bi-directional']), 'rel_ssr'].dropna()
    nonrel  = sub.loc[sub['granger_label'] == 'non-related', 'rel_ssr'].dropna()
    if len(causing) < 3 or len(nonrel) < 3:
        continue
    try:
        p = mannwhitneyu(causing, nonrel, alternative='two-sided').pvalue
    except ValueError:
        p = np.nan
    rows.append({
        'window_stratum': f"{int(stratum.left)}–{int(stratum.right)} pts",
        'median_window': int(sub['window_len'].median()),
        'n_causing': len(causing), 'n_nonrelated': len(nonrel),
        'median_relSSR_causing': round(causing.median(), 3),
        'median_relSSR_nonrelated': round(nonrel.median(), 3),
        'gap': round(nonrel.median() - causing.median(), 3),   # bigger = better separation
        'cliffs_delta': round(cliffs_delta(causing, nonrel), 3),
        'MWU_p': f"{p:.2e}",
    })

sep = pd.DataFrame(rows)
print("=== Separation of Granger-causing/bi-directional vs non-related, by window length ===")
print(sep.to_string(index=False))

# --- what fraction of each stratum gets a Granger-related label? ---
# if power drops with shorter windows, the detection RATE should drop too
rate = (d.assign(related=d['granger_label'].isin(['causing', 'bi-directional']))
          .groupby('wstratum', observed=True)
          .agg(n=('related', 'size'), n_related=('related', 'sum'),
               median_window=('window_len', 'median')))
rate['related_%'] = (100 * rate['n_related'] / rate['n']).round(1)
print("\n=== Detection rate of Granger-related labels, by window length ===")
print(rate.to_string())

# --- p-value distribution by stratum: flatter = less signal/power ---
print("\n=== Raw p (causing): fraction below 0.05, by window length ===")
pv = (d.assign(sig=d['p_causing'] < 0.05)
        .groupby('wstratum', observed=True)
        .agg(n=('sig', 'size'), n_sig=('sig', 'sum'),
             median_p=('p_causing', 'median')))
pv['sig_%'] = (100 * pv['n_sig'] / pv['n']).round(1)
pv['median_p'] = pv['median_p'].round(3)
print(pv.to_string())

# --- visualize: does the gap close as windows shorten? ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sep['median_window'], sep['gap'], marker='o', color='tab:blue')
axes[0].set_xlabel('median active window length (time points)')
axes[0].set_ylabel('median relSSR gap\n(non-related − causing)')
axes[0].set_title('Group separation vs. window length')
axes[0].axhline(0, color='gray', lw=0.8, ls=':')

axes[1].plot(sep['median_window'], sep['cliffs_delta'].abs(), marker='o', color='tab:purple')
axes[1].set_xlabel('median active window length (time points)')
axes[1].set_ylabel("|Cliff's δ|  (effect size)")
axes[1].set_title('Effect size vs. window length')
fig.tight_layout(); plt.show()

## histogram

In [ ]:
# ------------------------------------------------------------
# Distribution of active window lengths
# Bins colored by the same colormap used for the swarmplot dots.
# ------------------------------------------------------------
import matplotlib as mpl

CMAP = 'viridis'
full_len = len(log_interpolated_ts_df)
plt.rcParams.update({'font.size': 11, 'font.family': 'Arial'})

# same normalization as the swarmplots
vmin = int(perf_df['window_len'].min())
vmax = int(perf_df['window_len'].max())
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
cmap = mpl.colormaps[CMAP]

fig, ax = plt.subplots(figsize=(5, 3), dpi=800)

counts, bins, patches = ax.hist(perf_df['window_len'], bins=30)
for patch, left, right in zip(patches, bins[:-1], bins[1:]):
    patch.set_facecolor(cmap(norm((left + right) / 2)))
    patch.set_edgecolor('white')

ax.axvline(full_len, color='black', linestyle='--', linewidth=2,
          label=f'full series ({full_len} pts)')
ax.set_xlabel('Active window length (time points)')
ax.set_ylabel('Number of DVG candidates')
ax.set_title('Active window lengths')
ax.set_yscale('log')
ax.legend()

# colorbar, ticked at the true min and max
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_ticks([vmin, vmax])
cbar.set_ticklabels([str(vmin), str(vmax)])
cbar.set_label('Active window length (time points)', labelpad=10)

fig.tight_layout()
plt.show()

# ---- summary numbers for the text ----
print(perf_df.groupby('enrichment_type')['window_len']
              .agg(['count', 'min', 'median', 'max']).round(1))
print(f"\ncandidates with a restricted window (<= {MAX_POINTS} pts): "
      f"{int((perf_df['window_len'] <= MAX_POINTS).sum())} / {len(perf_df)}")
print(f"skipped entirely (too short / absent): {len(perf_errors)}")

## Sankey plots

In [ ]:
# ------------------------------------------------------------
# Genome-wide label flow: full time series -> active window
# corrected matched with corrected; uncorrected with uncorrected
# Ribbon HUE   = source (full-series) label
# Ribbon SHADE = active window completeness (darker = more complete)
# ------------------------------------------------------------
import plotly.graph_objects as go

LABELS = ['causing', 'bi-directional', 'caused', 'non-related']
DROP   = ['series_dropped_in_stationarization', 'window_too_short_or_absent']
ALL_ACTIVE = LABELS + DROP
COLORS = {'causing': '#1f77b4', 'bi-directional': '#9467bd', 'caused': '#d62728',
          'non-related': '#7f7f7f',
          'series_dropped_in_stationarization': '#c7c7c7',
          'window_too_short_or_absent': '#e0e0e0'}
PRETTY = {**{l: l for l in LABELS},
          'series_dropped_in_stationarization': 'dropped (not stationary)',
          'window_too_short_or_absent': 'excluded (window too short)'}

MIN_ALPHA, MAX_ALPHA = 0.15, 0.90     # short window -> faint, full series -> solid
plt.rcParams.update({'font.size': 12, 'font.family': 'Arial'})

def build_comparison(perf_df, perf_errors, full_labels_series, active_col='granger_label'):
    rows = {k: r[active_col] for k, r in perf_df.iterrows()}
    for k, reason in perf_errors.items():
        if k in rows: continue
        rows[k] = ('window_too_short_or_absent' if reason == 'window_too_short_or_absent'
                   else 'series_dropped_in_stationarization')
    active_label = pd.Series(rows, name='active_label')
    comp = pd.DataFrame({'full_label': full_labels_series}).join(active_label, how='left')
    comp['active_label'] = comp['active_label'].fillna('series_dropped_in_stationarization')
    return comp[comp['full_label'].isin(LABELS)]

def _shaded(hex_c, alpha):
    h = hex_c.lstrip('#'); r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha:.2f})"

def label_flow_plotly(comp, title):
    lidx = {l: i for i, l in enumerate(LABELS)}
    ridx = {l: i + len(LABELS) for i, l in enumerate(ALL_ACTIVE)}
    lc, rc_ = comp['full_label'].value_counts(), comp['active_label'].value_counts()
    node_labels = ([f"{l} ({int(lc.get(l,0))})" for l in LABELS] +
                   [f"{PRETTY[l]} ({int(rc_.get(l,0))})" for l in ALL_ACTIVE])
    node_colors = [COLORS[l] for l in LABELS] + [COLORS[l] for l in ALL_ACTIVE]

    flow = (comp.join(perf_df['window_len'])
                .groupby(['full_label', 'active_label'])
                .agg(n=('window_len', 'size'), w=('window_len', 'median'))
                .reset_index())

    # --- coloring: hue = source label, alpha = window completeness ---
    wmin = float(perf_df['window_len'].min())
    wmax = float(perf_df['window_len'].max())
    def _alpha(w):
        if pd.isna(w):                      # drop / excluded buckets: no window length
            return MIN_ALPHA
        frac = (w - wmin) / max(wmax - wmin, 1)
        return MIN_ALPHA + (MAX_ALPHA - MIN_ALPHA) * frac
    link_colors = [_shaded(COLORS[r.full_label], _alpha(r.w)) for r in flow.itertuples()]

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, color=node_colors, pad=16, thickness=20,
                  line=dict(color='white', width=0.5)),
        link=dict(source=[lidx[r.full_label] for r in flow.itertuples()],
                  target=[ridx[r.active_label] for r in flow.itertuples()],
                  value=[r.n for r in flow.itertuples()],
                  color=link_colors)))

    # --- shading legend (gray swatches, faint -> solid) ---
    for frac, lbl in [(0.0, f'{int(wmin)} pts (short)'),
                      (0.5, f'{int((wmin+wmax)/2)} pts'),
                      (1.0, f'{int(wmax)} pts (full series)')]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers', name=lbl,
            marker=dict(size=14, symbol='square',
                        color=_shaded('#555555', MIN_ALPHA + (MAX_ALPHA-MIN_ALPHA)*frac))))

    fig.update_layout(title_text=title, font=dict(size=13), height=620, width=1000,
                      showlegend=True,
                      legend=dict(title='Active window length<br>(darker = more complete)',
                                  x=1.02, y=0.5, bgcolor='rgba(255,255,255,0.8)'),
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig

full_uncorrected = summary_df[1].set_index('key')['plaque_assay_granger_label']
full_corrected   = summary_df_bh[1].set_index('key')['plaque_assay_granger_label']

for tag, full_src, active_col in [
        ('BH-corrected', full_corrected,   'granger_label'),
        ('uncorrected',  full_uncorrected, 'granger_label_raw')]:
    comp_v = build_comparison(perf_df, perf_errors, full_src, active_col=active_col)
    print(f"\n=== {tag} ===  compared: {len(comp_v)}")
    trans = (comp_v.groupby(['full_label', 'active_label']).size()
             .unstack(fill_value=0).reindex(index=LABELS, columns=ALL_ACTIVE, fill_value=0))
    print(trans)
    print(f"changed: {int((comp_v['full_label'] != comp_v['active_label']).sum())} / {len(comp_v)}")
    label_flow_plotly(comp_v, f"DVG label flow: full series → active window ({tag})").show()

    # which enrichment groups drive the changes
    ch = comp_v[comp_v['full_label'] != comp_v['active_label']].join(perf_df['enrichment_type'])
    print("\nchanged, by enrichment type:")
    print(ch.groupby(['enrichment_type', 'full_label', 'active_label']).size())

In [ ]:
# ------------------------------------------------------------
# Sankey (variant B): ribbons colored by the SAME viridis colormap
# used for the histogram / swarmplot dots -> ribbon color encodes the
# median active window length of the candidates in that flow.
# Node order fixed to: causing, bi-directional, caused, non-related.
# ------------------------------------------------------------
import numpy as np
import plotly.graph_objects as go
import matplotlib as mpl

CMAP_NAME = 'viridis'
_cmap = mpl.colormaps[CMAP_NAME]
_vmin = int(perf_df['window_len'].min())
_vmax = int(perf_df['window_len'].max())
_norm = mpl.colors.Normalize(vmin=_vmin, vmax=_vmax)

DROP_COLOR = 'rgba(200,200,200,0.45)'      # flows with no window length

NODE_ORDER = ['causing', 'bi-directional', 'caused', 'non-related']


def _rgba_from_cmap(w, alpha=0.55):
    if pd.isna(w):
        return DROP_COLOR
    r, g, b, _ = _cmap(_norm(w))
    return f"rgba({int(255*r)},{int(255*g)},{int(255*b)},{alpha})"


def _reorder(seq, order=NODE_ORDER):
    """Put `order` first (those present), keep any remaining items at the end."""
    seq = list(seq)
    return [l for l in order if l in seq] + [l for l in seq if l not in order]


def _yranks(counts, keys, pad=0.02):
    """Top-edge y spaced by each node's share of the column, so tall nodes fit."""
    v = np.array([counts.get(k, 0) for k in keys], dtype=float)
    if len(keys) == 1:
        return [0.5]
    if v.sum() == 0:
        return list(np.linspace(0.01, 0.99, len(keys)))
    frac = v / v.sum() * (1 - pad * (len(keys) - 1))
    tops = np.concatenate([[0.0], np.cumsum(frac[:-1] + pad)])
    return list(np.clip(tops, 0.001, 0.999))


def label_flow_viridis(comp, title, fixed=True):
    labels, all_active = _reorder(LABELS), _reorder(ALL_ACTIVE)
    lidx = {l: i for i, l in enumerate(labels)}
    ridx = {l: i + len(labels) for i, l in enumerate(all_active)}
    lc, rc_ = comp['full_label'].value_counts(), comp['active_label'].value_counts()
    node_labels = ([f"{l} ({int(lc.get(l,0))})" for l in labels] +
                   [f"{PRETTY[l]} ({int(rc_.get(l,0))})" for l in all_active])
    node_colors = [COLORS[l] for l in labels] + [COLORS[l] for l in all_active]

    flow = (comp.join(perf_df['window_len'])
                .groupby(['full_label', 'active_label'])
                .agg(n=('window_len', 'size'), w=('window_len', 'median'))
                .reset_index())

    node_kw = dict(label=node_labels, color=node_colors, pad=16, thickness=20,
                   line=dict(color='white', width=0.5))
    if fixed:
        node_kw['x'] = [0.001] * len(labels) + [0.999] * len(all_active)
        node_kw['y'] = _yranks(lc, labels) + _yranks(rc_, all_active)

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=node_kw,
        link=dict(source=[lidx[r.full_label] for r in flow.itertuples()],
                  target=[ridx[r.active_label] for r in flow.itertuples()],
                  value=[int(r.n) for r in flow.itertuples()],
                  color=[_rgba_from_cmap(r.w) for r in flow.itertuples()],
                  customdata=[('n/a' if pd.isna(r.w) else f"{r.w:.0f}")
                              for r in flow.itertuples()],
                  hovertemplate='%{value} DVGs<br>median window: %{customdata} pts<extra></extra>')))

    # continuous colorbar matching the histogram, ticked at true min/max
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers', showlegend=False,
        marker=dict(colorscale=CMAP_NAME, cmin=_vmin, cmax=_vmax,
                    color=[_vmin], size=0.1,
                    colorbar=dict(title='Median<br>active window<br>length (pts)',
                                  tickvals=[_vmin, _vmax],
                                  ticktext=[str(_vmin), str(_vmax)],
                                  len=1, x=1.02))))

    fig.update_layout(title_text=title, font=dict(size=18, color="black"),
                  height=640, width=1000,
                  xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig


full_corrected = summary_df_bh[1].set_index('key')['plaque_assay_granger_label']
comp_v = build_comparison(perf_df, perf_errors, full_corrected, active_col='granger_label')
label_flow_viridis(comp_v,
    '(A) DVG label flow: full series → active window (BH-corrected)').show()

In [ ]:
whole_ts_bh_df = summary_df_bh[1].copy()
whole_ts_bh_df['enrichment_type'] = whole_ts_bh_df['key'].apply(enrichment_type)
whole_ts_bh_df = whole_ts_bh_df.set_index('key')
whole_ts_bh_df['granger_label'] = whole_ts_bh_df['plaque_assay_granger_label']
whole_ts_bh_df.sort_values(by='key')

## Bar plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import Table2x2

ENRICHMENT_COLOR = {
    "loss":         "#f2c811",   # yellow
    "gain":         "#00c1c4",   # cyan
    "de novo gain": "#ec2d8a",   # magenta
    "de novo loss": "#656565",   # gray
}


def _lighten(color, amount=0.5):
    r, g, b = to_rgb(color)
    return (1 - amount * (1 - r), 1 - amount * (1 - g), 1 - amount * (1 - b))


def _stars(p, alpha=0.05):
    return "*" if p < alpha else "n.s."


def _effect_label(r, kind):
    if kind == "diff":
        return f"{r.pct_diff:+.0f} pp"
    if kind == "or":
        return f"OR {r.odds_ratio:.1f}"
    if kind == "or_ci":
        return f"OR {r.odds_ratio:.1f} [{r.or_lo:.1f}\u2013{r.or_hi:.1f}]"
    raise ValueError(kind)


def _pval_2x2(a, b, c, d, test="fisher", yates=True):
    """p-value for a 2x2 table. test='fisher' (exact) or 'chi2' (Pearson)."""
    tbl = [[a, b], [c, d]]
    if test == "fisher":
        return stats.fisher_exact(tbl)[1]
    if test == "chi2":
        arr = np.array(tbl, dtype=float)
        if (arr.sum(0) == 0).any() or (arr.sum(1) == 0).any():
            return np.nan            # empty margin: chi2 undefined
        return stats.chi2_contingency(arr, correction=yates)[1]
    raise ValueError(f"unknown sig_test: {test!r}")


def _contrasts_vs_reference(counts, reference, correction="fdr_bh",
                            test="fisher", yates=True):
    """Per (group, enrichment): test vs the same enrichment in `reference`."""
    rows = []
    for g in counts.index:
        if g == reference:
            continue
        for e in counts.columns:
            a = counts.loc[g, e]
            b = counts.loc[g].sum() - a
            c = counts.loc[reference, e]
            d = counts.loc[reference].sum() - c
            p = _pval_2x2(a, b, c, d, test=test, yates=yates)
            t = Table2x2(np.array([[a, b], [c, d]], float) + 0.5)  # Haldane-Anscombe
            lo, hi = t.oddsratio_confint()
            pct = 100 * a / (a + b) if (a + b) else np.nan
            pct_ref = 100 * c / (c + d) if (c + d) else np.nan
            rows.append(dict(group=g, enrichment=e, n=int(a + b),
                             pct=pct, pct_ref=pct_ref, pct_diff=pct - pct_ref,
                             min_expected=min(
                                 (a + b) * (a + c) / (a + b + c + d),
                                 (a + b) * (b + d) / (a + b + c + d),
                                 (c + d) * (a + c) / (a + b + c + d),
                                 (c + d) * (b + d) / (a + b + c + d)),
                             odds_ratio=t.oddsratio, or_lo=lo, or_hi=hi,
                             test=test, p=p))
    out = pd.DataFrame(rows)
    ok = out["p"].notna()
    out["p_adj"] = np.nan
    if ok.any():
        out.loc[ok, "p_adj"] = (multipletests(out.loc[ok, "p"], method=correction)[1]
                                if correction else out.loc[ok, "p"])
    return out


def plot_enrichment_by_granger(
    df,
    granger_col="granger_label",
    enrichment_col="enrichment_type",
    granger_order=None,
    enrichment_order=("gain", "loss", "de novo gain", "de novo loss"),
    colors=ENRICHMENT_COLOR,
    fractional=False,
    orientation="horizontal",
    invert_cat_axis=True,
    show_group_n="auto",
    group_n_fmt="{label}\n(n = {n})",
    sig_reference=None,
    sig_test="fisher",
    sig_yates=True,
    sig_correction="fdr_bh",
    sig_alpha=0.05,
    sig_show_ns=False,
    sig_offset=26,
    sig_fontsize=10,
    sig_headroom=1.18,
    sig_nudge=None,
    figsize=(8, 6),
    dpi=150,
    bar_width=0.8,
    lighten=0.5,
    edge_width=2,
    annotate=True,
    annot_fmt=None,
    annot_size=10,
    legend_loc="outside lower center",
    legend_bbox=None,
    legend_ncol=2,
    legend_mode=None,
    legend_kwargs=None,
    xlabel="Granger-causality label",
    ylabel=None,
    title=None,
    title_loc="auto",
    title_pad=None,
    title_kwargs=None,
    ax=None,
):
    """Grouped barplot of enrichment_type composition per granger_label.

    orientation='horizontal' draws barh (categories on the y axis).
    fractional=True gives within-group percentages instead of counts.
    sig_reference='non-related' marks each bar that differs from the same
    enrichment type in the reference group. sig_test='fisher' (two-sided
    exact) or 'chi2' (Pearson, Yates-corrected via sig_yates); both run on
    the raw counts, so marks are identical for count and fraction plots.
    Returns (fig, ax, table); table.attrs holds 'group_n' and 'contrasts'.
    """
    enrichment_order = list(enrichment_order)
    horiz = orientation.startswith("h")

    counts = (df.groupby([granger_col, enrichment_col])
                .size()
                .unstack(enrichment_col)
                .reindex(columns=enrichment_order)
                .fillna(0)
                .astype(int))
    if granger_order is not None:
        counts = counts.reindex(index=list(granger_order))
    group_n = counts.sum(axis=1).astype(int)

    contrasts = None
    if sig_reference is not None:
        if sig_reference not in counts.index:
            raise ValueError(f"sig_reference {sig_reference!r} not in {list(counts.index)}")
        contrasts = _contrasts_vs_reference(counts, sig_reference, sig_correction,
                                            test=sig_test, yates=sig_yates)

    table = counts.div(group_n, axis=0) * 100 if fractional else counts.astype(float)
    table.attrs["group_n"] = group_n
    table.attrs["contrasts"] = contrasts

    if show_group_n == "auto":
        show_group_n = fractional
    tick_labels = ([group_n_fmt.format(label=l, n=int(group_n.loc[l]))
                    for l in table.index]
                   if show_group_n else list(table.index))

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi, layout="constrained")
    else:
        fig = ax.figure

    pos = np.arange(len(table))
    width = bar_width / len(enrichment_order)
    if annot_fmt is None:
        annot_fmt = "%.1f" if fractional else "%d"

    for i, et in enumerate(enrichment_order):
        offset = (i - (len(enrichment_order) - 1) / 2) * width
        kw = dict(label=et,
                  facecolor=_lighten(colors[et], lighten),
                  edgecolor=colors[et], linewidth=edge_width)
        if horiz:
            bars = ax.barh(pos + offset, table[et].values, width, **kw)
        else:
            bars = ax.bar(pos + offset, table[et].values, width, **kw)
        if annotate:
            ax.bar_label(bars, fmt=annot_fmt, fontsize=annot_size, padding=2)

    if contrasts is not None:
        lookup = {(r.group, r.enrichment): r.p_adj for r in contrasts.itertuples()}
        for gi, g in enumerate(table.index):
            for i, et in enumerate(enrichment_order):
                p = lookup.get((g, et))
                if p is None or not np.isfinite(p):
                    continue
                mark = _stars(p, sig_alpha)
                if mark == "n.s." and not sig_show_ns:
                    continue
                nudge = (-0.32 * sig_fontsize if sig_nudge is None and mark == "*"
                         else (sig_nudge or 0.0))
                offset = (i - (len(enrichment_order) - 1) / 2) * width
                v = table.loc[g, et]
                xy = (v, gi + offset) if horiz else (gi + offset, v)
                xytext = (sig_offset, nudge) if horiz else (0, sig_offset + nudge)
                ax.annotate(mark, xy=xy, xytext=xytext, textcoords="offset points",
                            ha="left" if horiz else "center",
                            va="center" if horiz else "bottom",
                            fontsize=sig_fontsize)

    cat_label = xlabel
    val_label = (ylabel if ylabel is not None
                 else ("Fraction of group (%)" if fractional else "Number of candidates"))
    vmax = 105 if fractional else float(table.values.max()) * 1.05
    if contrasts is not None:
        vmax *= sig_headroom

    if horiz:
        ax.set_yticks(pos, tick_labels)
        ax.set_ylabel(cat_label)
        ax.set_xlabel(val_label)
        ax.set_xlim(0, vmax)
        if invert_cat_axis:
            ax.invert_yaxis()
    else:
        ax.set_xticks(pos, tick_labels)
        ax.set_xlabel(cat_label)
        ax.set_ylabel(val_label)
        ax.set_ylim(0, vmax)

    lk = dict(frameon=False, loc=legend_loc, ncol=legend_ncol, mode=legend_mode)
    if legend_bbox is not None:
        lk["bbox_to_anchor"] = legend_bbox
        lk["borderaxespad"] = 0.0
    lk.update(legend_kwargs or {})

    handles, lbls = ax.get_legend_handles_labels()
    if str(lk["loc"]).startswith("outside") and fig.get_layout_engine() is not None:
        lk.pop("bbox_to_anchor", None)
        fig.legend(handles, lbls, **lk)
    else:
        ax.legend(**lk)

    if title is not None:
        tk = dict(title_kwargs or {})
        use_fig = (title_loc == "figure" or
                   (title_loc == "auto" and str(lk["loc"]).startswith("outside upper")))
        if use_fig:
            fig.suptitle(title, **tk)
        else:
            ax.set_title(title, pad=title_pad, **tk)

    ax.spines[["top", "right"]].set_visible(False)
    if fig.get_layout_engine() is None:
        fig.tight_layout()
    return fig, ax, table

In [ ]:
plt.rcParams.update({'font.size': 12})
plot_enrichment_by_granger(whole_ts_bh_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=False, figsize=(3,5), title="Full time series (BH-corrected)")
plot_enrichment_by_granger(perf_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=False, figsize=(3,5), title="Active window (BH-corrected)")

### publication plot

In [ ]:
plt.rcParams.update({'font.size': 11})
plot_enrichment_by_granger(whole_ts_bh_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=True, figsize=(4,4), title="(B) Whole time series enrichment level fractions", sig_reference="non-related", sig_test='fisher')

plot_enrichment_by_granger(perf_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=True, figsize=(4,4), title="(C) Active window enrichment level fractions", sig_reference="non-related", sig_test='fisher')

### check enrichment type distribution with delta threshold

In [ ]:
def calculate_delta(dvg):
  if ts_df[dvg].iloc[0] == 0 and ts_df[dvg].iloc[-1] == 0:
    delta = ts_df[dvg].max()
  else:
    delta = ts_df[dvg].iloc[-1] - ts_df[dvg].iloc[0]
  return abs(delta)

whole_ts_bh_df["delta"] = whole_ts_bh_df.index.map(calculate_delta)
whole_ts_bh_df

slopes_df = pd.read_csv("data/datasets/pelz_2021/abs_slope_cluster_df.csv", index_col=0)

whole_ts_bh_df['abs_slope'] = whole_ts_bh_df.index.map(lambda dvg: abs(slopes_df.loc[slopes_df['key'] == dvg, 'slope'].values[0]) if dvg in slopes_df['key'].values else np.nan)

slope_cutoff90 = whole_ts_bh_df["abs_slope"].quantile(0.9)
slope_cutoff75 = whole_ts_bh_df["abs_slope"].quantile(0.75)
slope_cutoff50 = whole_ts_bh_df["abs_slope"].quantile(0.5)

print(slope_cutoff90, slope_cutoff75, slope_cutoff50)


In [ ]:

cutoff90 = whole_ts_bh_df["delta"].quantile(0.9)
cutoff75 = whole_ts_bh_df["delta"].quantile(0.75)
fig, ax = plt.subplots(figsize=(5, 3), dpi=300)
ax.axvline(cutoff90, color='black', linestyle='--', linewidth=1,
          label=f'90th percentile ({cutoff:.0f} pts)')
ax.axvline(cutoff75, color='gray', linestyle='--', linewidth=1,
          label=f'75th percentile ({cutoff:.0f} pts)')
whole_ts_bh_df["delta"].hist(bins=50, color="#1f77b4", edgecolor='white', ax=ax)

In [ ]:
filtered_whole_ts_bh_df = whole_ts_bh_df[whole_ts_bh_df["abs_slope"] >= slope_cutoff50].copy()
filtered_whole_ts_bh_df

In [ ]:
filtered_perf_df = perf_df[perf_df.index.isin(filtered_whole_ts_bh_df.index)].copy()

In [ ]:
plt.rcParams.update({'font.size': 11})
plot_enrichment_by_granger(filtered_whole_ts_bh_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=True, figsize=(4,4), title="(B) Whole time series enrichment level fractions", sig_reference="non-related", sig_test='fisher')

plot_enrichment_by_granger(filtered_perf_df, granger_order=['non-related', 'causing', 'bi-directional', 'caused'], fractional=True, figsize=(4,4), title="(C) Active window enrichment level fractions", sig_reference="non-related", sig_test='fisher')

## composition stats

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests


def composition_stats(df, granger_col="granger_label", enrichment_col="enrichment_type",
                      reference="non-related", alpha=0.05):
    """Omnibus test + adjusted residuals + 2x2 contrasts against a reference group."""
    tab = pd.crosstab(df[granger_col], df[enrichment_col])
    chi2, p_omni, dof, exp = stats.chi2_contingency(tab)
    n = tab.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))

    row = tab.sum(1).values[:, None] / n
    col = tab.sum(0).values[None, :] / n
    resid = pd.DataFrame((tab.values - exp) / np.sqrt(exp * (1 - row) * (1 - col)),
                         index=tab.index, columns=tab.columns)

    rows = []
    for g in tab.index.drop(reference):
        for e in tab.columns:
            a = tab.loc[g, e]; b = tab.loc[g].sum() - a
            c = tab.loc[reference, e]; d = tab.loc[reference].sum() - c
            odds, p = stats.fisher_exact([[a, b], [c, d]])
            rows.append(dict(group=g, enrichment=e, n_group=a + b,
                             pct_group=100 * a / (a + b),
                             pct_ref=100 * c / (c + d),
                             odds_ratio=odds, p=p))
    out = pd.DataFrame(rows)
    out["p_bh"] = multipletests(out["p"], method="fdr_bh")[1]
    out["sig"] = out["p_bh"] < alpha

    return dict(table=tab, chi2=chi2, dof=dof, p=p_omni, cramers_v=cramers_v,
                residuals=resid, contrasts=out.sort_values("p_bh"))


res = composition_stats(perf_df)
print(f"chi2 = {res['chi2']:.1f}, df = {res['dof']}, p = {res['p']:.2e}, "
      f"Cramér's V = {res['cramers_v']:.3f}")
res["contrasts"]

## for exploration

### Sankey label flow

In [ ]:
# ------------------------------------------------------------
# Sankey: enrichment (whole series) -> full label -> active label -> enrichment (active window)
# Ribbon colour = median active-window length (viridis, matching the histogram/swarmplot).
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib as mpl

cat_colors = {'loss': "#f2c811", 'gain': "#00c1c4", 'de_novo_gain': "#ec2d8a", 'de_novo_loss': "#656565"}

ENRICHMENT_ORDER = ["loss", "gain", "de novo gain", "de novo loss"]
ENRICHMENT_COLOR = {k.replace("_", " "): v for k, v in cat_colors.items()}
DROP_COLOR = "rgba(200,200,200,0.45)"


def label_flow_enrichment(
    comp,
    whole_ts_bh_df,
    perf_df,
    title,
    enrichment_col="enrichment_type",
    window_col="window_len",
    enrichment_order=ENRICHMENT_ORDER,
    enrichment_color=ENRICHMENT_COLOR,
    labels=None,               # defaults to LABELS
    all_active=None,           # defaults to ALL_ACTIVE
    pretty=None,               # defaults to PRETTY
    node_colors_granger=None,  # defaults to COLORS
    cmap_name="viridis",
    alpha=0.55,
    node_pad=10,
    node_thickness=20,
    x_positions=(0.001, 0.34, 0.66, 0.999),
    height=900,
    width=1250,
    font_size=15,
    margin=None,
    colorbar_x=1.02,
):
    """Four-column Sankey of DVG enrichment and Granger-causality label flow.

    `comp` must be indexed by key and carry 'full_label' and 'active_label'.
    Returns a plotly Figure.
    """
    labels = list(labels if labels is not None else LABELS)
    all_active = list(all_active if all_active is not None else ALL_ACTIVE)
    pretty = pretty if pretty is not None else PRETTY
    gcol = node_colors_granger if node_colors_granger is not None else COLORS
    enrichment_order = list(enrichment_order)
    margin = margin if margin is not None else dict(l=10, r=170, t=70, b=40)

    cmap = mpl.colormaps[cmap_name]
    vmin = int(perf_df[window_col].min())
    vmax = int(perf_df[window_col].max())
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    def rgba(w):
        if pd.isna(w):
            return DROP_COLOR
        r, g, b, _ = cmap(norm(w))
        return f"rgba({int(255*r)},{int(255*g)},{int(255*b)},{alpha})"

    keys = comp.index
    flow_df = pd.DataFrame({
        "enr_left":     whole_ts_bh_df[enrichment_col].reindex(keys),
        "full_label":   comp["full_label"],
        "active_label": comp["active_label"],
        "enr_right":    perf_df[enrichment_col].reindex(keys),
        "w":            perf_df[window_col].reindex(keys),
    })

    # --- node blocks -------------------------------------------------------
    n_e, n_l, n_a = len(enrichment_order), len(labels), len(all_active)
    i_el = {e: i for i, e in enumerate(enrichment_order)}
    i_fl = {l: n_e + i for i, l in enumerate(labels)}
    i_al = {l: n_e + n_l + i for i, l in enumerate(all_active)}
    i_er = {e: n_e + n_l + n_a + i for i, e in enumerate(enrichment_order)}

    cl = flow_df["enr_left"].value_counts()
    cf = flow_df["full_label"].value_counts()
    ca = flow_df["active_label"].value_counts()
    cr = flow_df["enr_right"].value_counts()

    node_labels = ([f"{e} ({int(cl.get(e, 0))})" for e in enrichment_order] +
                   [f"{l} ({int(cf.get(l, 0))})" for l in labels] +
                   [f"{pretty[l]} ({int(ca.get(l, 0))})" for l in all_active] +
                   [f"{e} ({int(cr.get(e, 0))})" for e in enrichment_order])
    node_colors = ([enrichment_color[e] for e in enrichment_order] +
                   [gcol[l] for l in labels] +
                   [gcol[l] for l in all_active] +
                   [enrichment_color[e] for e in enrichment_order])

    block_sizes = [n_e, n_l, n_a, n_e]
    node_x = []
    for xpos, k in zip(x_positions, block_sizes):
        node_x += [xpos] * k

    # --- links -------------------------------------------------------------
    def stage(src_col, tgt_col, src_idx, tgt_idx):
        g = (flow_df.dropna(subset=[src_col, tgt_col])
                    .groupby([src_col, tgt_col], observed=True)
                    .agg(n=("w", "size"), w=("w", "median"))
                    .reset_index())
        g["pct_src"] = g["n"] / g.groupby(src_col, observed=True)["n"].transform("sum") * 100
        g["pct_tgt"] = g["n"] / g.groupby(tgt_col, observed=True)["n"].transform("sum") * 100
        return ([src_idx[r[0]] for r in g.itertuples(index=False)],
                [tgt_idx[r[1]] for r in g.itertuples(index=False)],
                [int(r.n) for r in g.itertuples()],
                [r.w for r in g.itertuples()],
                [(r.pct_src, r.pct_tgt) for r in g.itertuples()])

    src, tgt, val, med, pct = [], [], [], [], []
    for args in [("enr_left", "full_label", i_el, i_fl),
                 ("full_label", "active_label", i_fl, i_al),
                 ("active_label", "enr_right", i_al, i_er)]:
        s, t, v, w, p = stage(*args)
        src += s; tgt += t; val += v; med += w; pct += p

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(label=node_labels, color=node_colors, x=node_x,
                  pad=node_pad, thickness=node_thickness,
                  line=dict(color="white", width=0.5)),
        link=dict(source=src, target=tgt, value=val,
                  color=[rgba(w) for w in med],
                  customdata=[["n/a" if pd.isna(w) else f"{w:.0f}", f"{ps:.1f}", f"{pt:.1f}"]
                              for w, (ps, pt) in zip(med, pct)],
                  hovertemplate=("%{value} DVGs<br>"
                                "%{customdata[1]}% of %{source.label}<br>"
                                "%{customdata[2]}% of %{target.label}<br>"
                                "median window: %{customdata[0]} pts"
                                "<extra></extra>"))))
    
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", showlegend=False,
        marker=dict(colorscale=cmap_name, cmin=vmin, cmax=vmax, color=[vmin], size=0.1,
                    colorbar=dict(title="Median<br>active window<br>length (pts)",
                                  tickvals=[vmin, vmax], ticktext=[str(vmin), str(vmax)],
                                  len=1, x=colorbar_x))))

    fig.update_layout(title_text=title, font=dict(size=font_size),
                      height=height, width=width, margin=margin,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig


label_flow_enrichment(
    comp_v, whole_ts_bh_df, perf_df,
    "DVG enrichment → label flow: full series → active window (BH-corrected)"
).show()

In [ ]:
# ------------------------------------------------------------
# Sankey: enrichment (whole series) -> full label -> active label -> enrichment (active window)
# Ribbon colour = median active-window length (viridis, matching the histogram/swarmplot).
# Granger node order fixed to: causing, bi-directional, caused, non-related.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib as mpl

cat_colors = {'loss': "#f2c811", 'gain': "#00c1c4", 'de_novo_gain': "#ec2d8a", 'de_novo_loss': "#656565"}

ENRICHMENT_ORDER = ["loss", "gain", "de novo gain", "de novo loss"]
ENRICHMENT_COLOR = {k.replace("_", " "): v for k, v in cat_colors.items()}
DROP_COLOR = "rgba(200,200,200,0.45)"

NODE_ORDER = ['causing', 'bi-directional', 'caused', 'non-related']

def label_flow_enrichment(
    comp,
    whole_ts_bh_df,
    perf_df,
    title,
    enrichment_col="enrichment_type",
    window_col="window_len",
    enrichment_order=ENRICHMENT_ORDER,
    enrichment_color=ENRICHMENT_COLOR,
    labels=None,               # defaults to LABELS
    all_active=None,           # defaults to ALL_ACTIVE
    pretty=None,               # defaults to PRETTY
    node_colors_granger=None,  # defaults to COLORS
    node_order=NODE_ORDER,
    fixed=True,
    y_pad=0.02,
    cmap_name="viridis",
    alpha=0.55,
    node_pad=10,
    node_thickness=20,
    x_positions=(0.001, 0.34, 0.66, 0.999),
    height=900,
    width=1250,
    font_size=15,
    margin=None,
    colorbar_x=1.02,
):
    """Four-column Sankey of DVG enrichment and Granger label flow.

    `comp` must be indexed by key and carry 'full_label' and 'active_label'.
    Returns a plotly Figure.
    """
    labels = _reorder(labels if labels is not None else LABELS, node_order)
    all_active = _reorder(all_active if all_active is not None else ALL_ACTIVE, node_order)
    pretty = pretty if pretty is not None else PRETTY
    gcol = node_colors_granger if node_colors_granger is not None else COLORS
    enrichment_order = list(enrichment_order)
    margin = margin if margin is not None else dict(l=10, r=170, t=70, b=40)

    cmap = mpl.colormaps[cmap_name]
    vmin = int(perf_df[window_col].min())
    vmax = int(perf_df[window_col].max())
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    def rgba(w):
        if pd.isna(w):
            return DROP_COLOR
        r, g, b, _ = cmap(norm(w))
        return f"rgba({int(255*r)},{int(255*g)},{int(255*b)},{alpha})"

    keys = comp.index
    flow_df = pd.DataFrame({
        "enr_left":     whole_ts_bh_df[enrichment_col].reindex(keys),
        "full_label":   comp["full_label"],
        "active_label": comp["active_label"],
        "enr_right":    perf_df[enrichment_col].reindex(keys),
        "w":            perf_df[window_col].reindex(keys),
    })

    # --- node blocks -------------------------------------------------------
    n_e, n_l, n_a = len(enrichment_order), len(labels), len(all_active)
    i_el = {e: i for i, e in enumerate(enrichment_order)}
    i_fl = {l: n_e + i for i, l in enumerate(labels)}
    i_al = {l: n_e + n_l + i for i, l in enumerate(all_active)}
    i_er = {e: n_e + n_l + n_a + i for i, e in enumerate(enrichment_order)}

    cl = flow_df["enr_left"].value_counts()
    cf = flow_df["full_label"].value_counts()
    ca = flow_df["active_label"].value_counts()
    cr = flow_df["enr_right"].value_counts()

    node_labels = ([f"{e} ({int(cl.get(e, 0))})" for e in enrichment_order] +
                   [f"{l} ({int(cf.get(l, 0))})" for l in labels] +
                   [f"{pretty[l]} ({int(ca.get(l, 0))})" for l in all_active] +
                   [f"{e} ({int(cr.get(e, 0))})" for e in enrichment_order])
    node_colors = ([enrichment_color[e] for e in enrichment_order] +
                   [gcol[l] for l in labels] +
                   [gcol[l] for l in all_active] +
                   [enrichment_color[e] for e in enrichment_order])

    block_sizes = [n_e, n_l, n_a, n_e]
    node_x = []
    for xpos, k in zip(x_positions, block_sizes):
        node_x += [xpos] * k
    node_y = (_yranks(cl, enrichment_order, y_pad) +
              _yranks(cf, labels, y_pad) +
              _yranks(ca, all_active, y_pad) +
              _yranks(cr, enrichment_order, y_pad))

    # --- links -------------------------------------------------------------
    def stage(src_col, tgt_col, src_idx, tgt_idx):
        g = (flow_df.dropna(subset=[src_col, tgt_col])
                    .groupby([src_col, tgt_col], observed=True)
                    .agg(n=("w", "size"), w=("w", "median"))
                    .reset_index())
        g["pct_src"] = g["n"] / g.groupby(src_col, observed=True)["n"].transform("sum") * 100
        g["pct_tgt"] = g["n"] / g.groupby(tgt_col, observed=True)["n"].transform("sum") * 100
        return ([src_idx[r[0]] for r in g.itertuples(index=False)],
                [tgt_idx[r[1]] for r in g.itertuples(index=False)],
                [int(r.n) for r in g.itertuples()],
                [r.w for r in g.itertuples()],
                [(r.pct_src, r.pct_tgt) for r in g.itertuples()])

    src, tgt, val, med, pct = [], [], [], [], []
    for args in [("enr_left", "full_label", i_el, i_fl),
                 ("full_label", "active_label", i_fl, i_al),
                 ("active_label", "enr_right", i_al, i_er)]:
        s, t, v, w, p = stage(*args)
        src += s; tgt += t; val += v; med += w; pct += p

    node_kw = dict(label=node_labels, color=node_colors,
                   pad=node_pad, thickness=node_thickness,
                   line=dict(color="white", width=0.5))
    if fixed:
        node_kw["x"] = node_x
        node_kw["y"] = node_y

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=node_kw,
        link=dict(source=src, target=tgt, value=val,
                  color=[rgba(w) for w in med],
                  customdata=[["n/a" if pd.isna(w) else f"{w:.0f}", f"{ps:.1f}", f"{pt:.1f}"]
                              for w, (ps, pt) in zip(med, pct)],
                  hovertemplate=("%{value} DVGs<br>"
                                 "%{customdata[1]}% of %{source.label}<br>"
                                 "%{customdata[2]}% of %{target.label}<br>"
                                 "median window: %{customdata[0]} pts"
                                 "<extra></extra>"))))

    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", showlegend=False,
        marker=dict(colorscale=cmap_name, cmin=vmin, cmax=vmax, color=[vmin], size=0.1,
                    colorbar=dict(title="Median<br>active window<br>length (pts)",
                                  tickvals=[vmin, vmax], ticktext=[str(vmin), str(vmax)],
                                  len=1, x=colorbar_x))))

    fig.update_layout(title_text=title, font=dict(size=font_size),
                      height=height, width=width, margin=margin,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig

label_flow_enrichment(
    comp_v, whole_ts_bh_df, perf_df,
    "DVG enrichment → label flow: full series → active window (BH-corrected)"
).show()

### tabular label flow

In [ ]:
# ------------------------------------------------------------
# Tabular flow summary
# ------------------------------------------------------------
c = comp_v.join(perf_df[['window_len', 'enrichment_type']])

# (1) transition matrix: counts
trans = (c.groupby(['full_label', 'active_label']).size()
           .unstack(fill_value=0)
           .reindex(index=LABELS, columns=ALL_ACTIVE, fill_value=0))
trans['TOTAL'] = trans.sum(axis=1)
print("=== Transition counts (rows = full series, cols = active window) ===")
print(trans.to_string())

# (2) same, as row percentages
pct = (trans[ALL_ACTIVE].div(trans['TOTAL'], axis=0) * 100).round(1)
print("\n=== Transition percentages (% of each full-series group) ===")
print(pct.to_string())

# (3) per-flow detail: n, median window, retained vs changed
detail = (c.groupby(['full_label', 'active_label'])
            .agg(n=('window_len', 'size'),
                 median_window=('window_len', 'median'),
                 min_window=('window_len', 'min'),
                 max_window=('window_len', 'max'))
            .reset_index())
detail['changed'] = detail['full_label'] != detail['active_label']
detail = detail.sort_values(['full_label', 'n'], ascending=[True, False])
print("\n=== Flow detail ===")
print(detail.to_string(index=False))

# (4) stability per group
stab = (c.assign(same=lambda d: d['full_label'] == d['active_label'])
          .groupby('full_label')
          .agg(n=('same', 'size'), retained=('same', 'sum')))
stab['changed'] = stab['n'] - stab['retained']
stab['retained_%'] = (100 * stab['retained'] / stab['n']).round(1)
print("\n=== Label stability by full-series group ===")
print(stab.to_string())

# (5) the directional asymmetry check
c['gained'] = c['full_label'].eq('non-related') & c['active_label'].isin(['causing', 'bi-directional'])
c['lost']   = c['full_label'].isin(['causing', 'bi-directional']) & c['active_label'].eq('non-related')
wbin = pd.qcut(c['window_len'], 4, duplicates='drop')
print("\n=== Gained vs lost Granger-relatedness, by window-length quartile ===")
print(c.groupby(wbin, observed=True)[['gained', 'lost']].sum().to_string())
print(f"\nTOTAL gained: {int(c['gained'].sum())}   TOTAL lost: {int(c['lost'].sum())}")

In [ ]:
c['gained'] = c['full_label'].eq('non-related') & c['active_label'].isin(['causing','bi-directional'])
c['lost']   = c['full_label'].isin(['causing','bi-directional']) & c['active_label'].eq('non-related')
print(f"gained: {int(c['gained'].sum())}   lost: {int(c['lost'].sum())}")

In [ ]:
# ------------------------------------------------------------
# Sankey with a two-way intermediate column:
#   full-series label -> {full time series | partial time series} -> active-window label
# ------------------------------------------------------------
import plotly.graph_objects as go

LABELS = ['causing', 'bi-directional', 'caused', 'non-related']
LABEL_COLORS = {'causing': '#1f77b4', 'bi-directional': '#9467bd',
                'caused': '#d62728', 'non-related': '#7f7f7f'}

FULL_COLOR    = '#2c7a4b'   # full time series
PARTIAL_COLOR = '#e07b39'   # partial time series
UNKNOWN_COLOR = '#b0b0b0'   # dropped / excluded, no window_len

def _rgba_from_hex(hex_c, alpha=0.55):
    r, g, b = [int(hex_c.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)]
    return f"rgba({r},{g},{b},{alpha})"

def sankey_full_vs_partial(perf_df, perf_errors, full_labels_series,
                           active_col='granger_label',
                           title='Label flow: full vs. partial active window'):
    full_len = len(log_interpolated_ts_df)

    d = perf_df.copy()
    d['full_label'] = full_labels_series.reindex(d.index)
    d = d.dropna(subset=['full_label'])
    d = d[d['full_label'].isin(LABELS)]

    d['window_group'] = np.where(d['window_len'] >= full_len,
                                 'full time series', 'partial time series')

    # candidates that couldn't be tested at all (no window_len) -> explicit bucket
    missing_keys = [k for k in perf_errors if k in d.index or
                    (full_labels_series.get(k) in LABELS)]
    # rebuild including candidates present in full_labels_series but absent from perf_df
    all_idx = full_labels_series[full_labels_series.isin(LABELS)].index
    extra = [k for k in all_idx if k not in d.index]
    if extra:
        extra_df = pd.DataFrame({'full_label': full_labels_series.reindex(extra)})
        extra_df['window_group'] = 'excluded / not tested'
        extra_df[active_col] = extra_df.index.map(
            lambda k: 'excluded / not tested')
        d = pd.concat([d[['full_label', 'window_group', active_col]], extra_df], axis=0)

    groups = ['full time series', 'partial time series', 'excluded / not tested']
    group_colors = {'full time series': FULL_COLOR,
                    'partial time series': PARTIAL_COLOR,
                    'excluded / not tested': UNKNOWN_COLOR}

    right_labels = LABELS + (['excluded / not tested']
                             if 'excluded / not tested' in d['window_group'].unique() else [])

    L = {l: i for i, l in enumerate(LABELS)}
    G = {g: len(LABELS) + i for i, g in enumerate(groups)}
    R = {l: len(LABELS) + len(groups) + i for i, l in enumerate(right_labels)}

    lc = d['full_label'].value_counts()
    gc = d['window_group'].value_counts()
    rc = d[active_col].value_counts()

    node_labels = ([f"{l} ({int(lc.get(l,0))})" for l in LABELS] +
                   [f"{g} ({int(gc.get(g,0))})" for g in groups] +
                   [f"{l} ({int(rc.get(l,0))})" for l in right_labels])
    node_colors = ([LABEL_COLORS[l] for l in LABELS] +
                   [group_colors[g] for g in groups] +
                   [LABEL_COLORS.get(l, UNKNOWN_COLOR) for l in right_labels])

    src, tgt, val, col = [], [], [], []
    # stage 1: full label -> window group (colored by source label)
    for (fl, g), n in d.groupby(['full_label', 'window_group'], observed=True).size().items():
        if n == 0: continue
        src.append(L[fl]); tgt.append(G[g]); val.append(int(n))
        col.append(_rgba_from_hex(LABEL_COLORS[fl]))
    # stage 2: window group -> active label (colored by window group)
    for (g, al), n in d.groupby(['window_group', active_col], observed=True).size().items():
        if n == 0: continue
        src.append(G[g]); tgt.append(R[al]); val.append(int(n))
        col.append(_rgba_from_hex(group_colors[g]))

    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, color=node_colors, pad=16, thickness=20,
                  line=dict(color='white', width=0.5)),
        link=dict(source=src, target=tgt, value=val, color=col)))
    fig.update_layout(title_text=title, font=dict(size=13), height=650, width=1000)
    return fig

full_corrected = summary_df_bh[1].set_index('key')['plaque_assay_granger_label']
sankey_full_vs_partial(perf_df, perf_errors, full_corrected, 'granger_label',
    title='Full-series label → window type → active-window label (BH-corrected)').show()

In [ ]:
# ------------------------------------------------------------
# Sankey with every distinct active window size as an intermediate column,
# EXPLICITLY sorted top-to-bottom by window length.
# Middle nodes colored by viridis (same mapping as histogram/swarmplots).
# ------------------------------------------------------------
import plotly.graph_objects as go
import matplotlib as mpl
import numpy as np

LABELS = ['causing', 'bi-directional', 'caused', 'non-related']
LABEL_COLORS = {'causing': '#1f77b4', 'bi-directional': '#9467bd',
                'caused': '#d62728', 'non-related': '#7f7f7f'}
CMAP = mpl.colormaps['viridis']

def _rgba(hex_or_rgba, a=0.55):
    if isinstance(hex_or_rgba, str):
        r, g, b = [int(hex_or_rgba.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)]
    else:
        r, g, b = [int(255 * v) for v in hex_or_rgba[:3]]
    return f"rgba({r},{g},{b},{a})"

def sankey_by_window_size(perf_df, full_labels_series, active_col='granger_label',
                          title='Label flow by active window size'):
    d = perf_df.copy()
    d['full_label'] = full_labels_series.reindex(d.index)
    d = d.dropna(subset=['full_label'])
    d = d[d['full_label'].isin(LABELS) & d[active_col].isin(LABELS)]
    d['window_len'] = d['window_len'].astype(int)

    sizes = sorted(d['window_len'].unique(), reverse=True)   # descending: longest (top) -> shortest (bottom)
    vmin, vmax = min(sizes), max(sizes)
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    size_rgba = {s: CMAP(norm(s)) for s in sizes}

    n_left, n_mid, n_right = len(LABELS), len(sizes), len(LABELS)
    L = {l: i for i, l in enumerate(LABELS)}
    S = {s: n_left + i for i, s in enumerate(sizes)}          # index order == sorted order
    R = {l: n_left + n_mid + i for i, l in enumerate(LABELS)}

    lc = d['full_label'].value_counts()
    sc = d['window_len'].value_counts()
    rc = d[active_col].value_counts()

    node_labels = ([f"{l} ({int(lc.get(l,0))})" for l in LABELS] +
                   [f"{s} pts ({int(sc.get(s,0))})" for s in sizes] +
                   [f"{l} ({int(rc.get(l,0))})" for l in LABELS])
    node_colors = ([LABEL_COLORS[l] for l in LABELS] +
                   [_rgba(size_rgba[s], 0.95) for s in sizes] +
                   [LABEL_COLORS[l] for l in LABELS])

    # --- explicit node positions (arrangement='fixed') ---
    def col_positions(n):
        if n == 1:
            return [0.5]
        return list(np.linspace(0.06, 0.92, n))       # was 0.06 -> starts lower

    x = [0.15] * n_left + [0.5] * n_mid + [0.85] * n_right
    y = col_positions(n_left) + col_positions(n_mid) + col_positions(n_right)

    src, tgt, val, col = [], [], [], []
    for (fl, s), n in d.groupby(['full_label', 'window_len'], observed=True).size().items():
        src.append(L[fl]); tgt.append(S[s]); val.append(int(n))
        col.append(_rgba(LABEL_COLORS[fl]))
    for (s, al), n in d.groupby(['window_len', active_col], observed=True).size().items():
        src.append(S[s]); tgt.append(R[al]); val.append(int(n))
        col.append(_rgba(size_rgba[s]))

    fig = go.Figure(go.Sankey(
        arrangement='fixed',
        node=dict(label=node_labels, color=node_colors, pad=8, thickness=18,
                  line=dict(color='white', width=0.5), x=x, y=y),
        link=dict(source=src, target=tgt, value=val, color=col)))

    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers', showlegend=False,
        marker=dict(colorscale='viridis', cmin=vmin, cmax=vmax, color=[vmin], size=0.1,
                    colorbar=dict(title='Active window<br>length (pts)',
                                  tickvals=[vmin, vmax], ticktext=[str(vmin), str(vmax)],
                                  len=1, x=0.9))))

    fig.update_layout(
        title=dict(text=title, x=0.2, xanchor='left', y=0.97, yanchor='top',
                   font=dict(size=15)),
        font=dict(size=12),
        height=max(500, 26 * n_mid),
        width=1250,
        margin=dict(l=140, r=200, t=140, b=120),
        paper_bgcolor='white',
        plot_bgcolor='rgba(0,0,0,0)',
        xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig

full_corrected = summary_df_bh[1].set_index('key')['plaque_assay_granger_label']
sankey_by_window_size(perf_df, full_corrected, 'granger_label',
    title='Full-series label → active window size → active-window label (BH-corrected)').show()

## experimentally validated candidates (publication panel)

In [ ]:
# ============================================================
# Experimentally validated candidates: active-window analysis + OLS fit plots
#
# Reuses the SAME functions as the genome-wide active-window analysis
# (enrichment, active_window_df) so windows and fits cannot drift apart.
#
# NOTE: labels here are assigned from UNCORRECTED p-values, as these three
# candidates are a pre-specified confirmatory analysis (see Methods), not a
# screen. The genome-wide analysis uses BH-corrected labels.
# ============================================================
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

# ------------------------------------------------------------------ STYLE CONFIG
# One place to control the look of every candidate plot. Matches the
# run_granger_prediction() publication style.
STYLE = {
    'figsize':          (5, 3),
    'dpi':              800,
    'linewidth':        3,
    'markersize':       8,          # restricted + full markers
    'markeredgewidth':  1,
    'ylim':             (0, 10),
    'x_major':          2,          # major x-tick spacing
    'x_minor':          2,          # minor ticks between majors
    'ylabel':           'log10(PFU/mL+1)',
    'xlabel':           'Time post infection (days)',
    'ylabel_y':         -0.01,       # y-position of the y-label (bottom-aligned)
    'ylabel_ha':        'left',

    # measured / data trace
    'meas_marker':      'D',
    'meas_markersize':  7,          # set to None -> use markersize-1
    'meas_color':       'black',

    # restricted model
    'restr_marker':     '^',
    'restr_color':      '#e09312',
    'restr_facecolor':  '#e1bf83',
    'restr_linestyle':  (0, (1, 1)),

    # full model
    'full_marker':      'o',
    'full_linestyle':   'dashed',
    'full_facelighten': 0.5,        # lighten_color amount for the marker face

    # legend
    'legend_ncol':      1,
    'legend_frameon':   False,
    'legend_loc':       'center left',
    'legend_bbox':      (1.05, 0.6),
}

VALIDATED = {
    'PB2_217_2204': 'gain',
    'PB2_129_2176': 'loss',
    'PB2_269_2202': 'de novo gain',
}

def fit_candidate(key, window_df, opt_lag=OPT_LAG):
    """Fit restricted + full OLS on a window. Identical preprocessing to
    active_window_performance(): PFU undifferenced, DVG stationarized."""
    dvg_res = make_stationary(window_df[[key]], max_diff=10, autolag='AIC', max_lag=None)
    if key not in dvg_res.df_stationary.columns:
        raise ValueError(f'{key}: DVG series dropped during stationarization')
    d_dvg = int(dvg_res.diff_levels[key]) if np.isfinite(dvg_res.diff_levels[key]) else 0
    dvg_stat = dvg_res.df_stationary[key].fillna(0)
    pfu = window_df['plaque_assay'].astype(float)

    gc_df = pd.concat([pfu.rename('plaque_assay'), dvg_stat.rename(key)], axis=1).dropna()
    gc_df = gc_df.astype('float64')
    if len(gc_df) < opt_lag + 2:
        raise ValueError(f'{key}: too few points (n={len(gc_df)})')

    causing = grangercausalitytests(gc_df[['plaque_assay', key]], maxlag=1, verbose=False)
    caused  = grangercausalitytests(gc_df[[key, 'plaque_assay']], maxlag=1, verbose=False)

    p_causing = causing[1][0]['ssr_chi2test'][1]
    p_caused  = caused[1][0]['ssr_chi2test'][1]
    full_res, restr_res = causing[1][1][1], causing[1][1][0]

    return {'key': key,
            'label': classify_from_pvals(p_causing, p_caused),   # uncorrected
            'p_causing': p_causing, 'p_caused': p_caused,
            'ssr_restricted': float(restr_res.ssr),
            'ssr_full': float(full_res.ssr),
            'restr_pred': restr_res.predict(),
            'full_pred': full_res.predict(),
            'dpis': gc_df.index.values,
            'window_start': float(window_df.index.min()),
            'window_end': float(window_df.index.max()),
            'n': len(gc_df), 'diff_dvg': d_dvg}

def plot_candidate_fit(fit, series_label, show_title=True,
                       show_legend=True, show_ssr=True, style=None):
    """Plot measured PFU + restricted/full OLS fits, in the run_granger_prediction style.
    Pass style=... to override the module-level STYLE dict for a single call."""
    s = {**STYLE, **(style or {})}
    key, label, opt_lag = fit['key'], fit['label'], OPT_LAG
    full_color = granger_label_color_map[label]
    dpis = fit['dpis']
    meas_ms = s['meas_markersize'] if s['meas_markersize'] is not None \
              else max(1, s['markersize'] - 1)

    fig, ax = plt.subplots(figsize=s['figsize'], dpi=s['dpi'])

    # measured PFU across the WHOLE cultivation, for visual context
    ax.plot(log_interpolated_ts_df.index.values,
            log_interpolated_ts_df['plaque_assay'].astype(float).values,
            label='Data', marker=s['meas_marker'], markersize=meas_ms,
            color=s['meas_color'], linewidth=s['linewidth'])
    ax.plot(dpis[opt_lag:], fit['restr_pred'], label='Restricted\nmodel',
            marker=s['restr_marker'], markersize=s['markersize'],
            color=s['restr_color'], markeredgecolor=s['restr_color'],
            markerfacecolor=s['restr_facecolor'], markeredgewidth=s['markeredgewidth'],
            linewidth=s['linewidth'], linestyle=s['restr_linestyle'])
    ax.plot(dpis[opt_lag:], fit['full_pred'], label='Full\nmodel',
            marker=s['full_marker'], markersize=s['markersize'],
            color=full_color, markeredgecolor=full_color,
            markerfacecolor=lighten_color(full_color, s['full_facelighten']),
            markeredgewidth=s['markeredgewidth'],
            linewidth=s['linewidth'], linestyle=s['full_linestyle'])

    if show_title:
        ax.set_title(f'{key} ({VALIDATED[key]}) — {series_label}\nGranger-{label} (lag={opt_lag})')
    ax.set_xlabel(s['xlabel'])
    ax.set_ylabel(s['ylabel'], y=s['ylabel_y'], ha=s['ylabel_ha'])
    ax.xaxis.set_major_locator(MultipleLocator(s['x_major']))
    ax.xaxis.set_minor_locator(AutoMinorLocator(s['x_minor']))
    ax.set_ylim(*s['ylim'])
    ax.set_yscale('linear')

    if show_legend:
        ax.legend(loc=s['legend_loc'], bbox_to_anchor=s['legend_bbox'],
                  ncol=s['legend_ncol'], frameon=s['legend_frameon'])
    if show_ssr:
        fig.text(1.1, 0.10, f"SSR$_{{restricted}}$={fit['ssr_restricted']:.2f}",
                 transform=ax.transAxes, verticalalignment='top',
                 bbox=dict(boxstyle='square', facecolor=s['restr_color'], alpha=0.5))
        fig.text(1.1, -0.05, f"SSR$_{{full}}$={fit['ssr_full']:.2f}",
                 transform=ax.transAxes, verticalalignment='top',
                 bbox=dict(boxstyle='square', facecolor=full_color, alpha=0.5))
    plt.show()
    return fig, ax

# ---- run each candidate on BOTH the full series and its active window ----
rows = []
for key, etype in VALIDATED.items():
    full_win   = log_interpolated_ts_df[['plaque_assay', key]].copy()
    active_win = active_window_df(key, enrichment[key])          # SHARED definition

    is_restricted = len(active_win) < len(full_win)

    for series_label, win in [('whole time series', full_win)] + \
                             ([('active window', active_win)] if is_restricted else []):
        fit = fit_candidate(key, win)
        plot_candidate_fit(fit, series_label)
        rows.append({'key': key, 'enrichment': etype, 'series': series_label,
                     'window': f"{fit['window_start']:.2f}–{fit['window_end']:.2f}",
                     'n': fit['n'], 'label': fit['label'],
                     'p_causing': fit['p_causing'], 'p_caused': fit['p_caused'],
                     'SSR_restricted': fit['ssr_restricted'],
                     'SSR_full': fit['ssr_full']})

validated_df = pd.DataFrame(rows)
print(validated_df.to_string(index=False))

# sanity: the shared enrichment call should agree with the known types
print("\nenrichment assigned by the general analysis:")
print({k: enrichment[k] for k in VALIDATED})

In [ ]:
# ============================================================
# Individual plot elements for the results panel figure
# Produces, separately:
#   - one bare plot per (candidate x series)  [no title / legend / SSR boxes]
#   - a standalone shared legend
#   - a text/CSV dump of labels, SSRs and p-values for typesetting
# ============================================================
import os
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.lines import Line2D

PANEL_DIR = f'{output_prefix}/plots/panel_elements'
os.makedirs(PANEL_DIR, exist_ok=True)
plt.rcParams.update({'font.size': 12, 'font.family': 'Arial'})

# per-enrichment panel size (inches): gain is the wide format, the other two square-ish
FIG_SIZES = {
    'gain':         (4.72,1.5),
    'loss':         (3, 2),
    'de novo gain': (3, 2),
}

# ------------------------------------------------------------------ STYLE CONFIG
# One place to control the look of every panel element. Keep in sync with the
# STYLE dict used for plot_candidate_fit / run_granger_prediction.

PANEL_STYLE = {
    'figsize':          (3.5, 2),       # default; overridden per enrichment via FIG_SIZES
    'dpi':              800,
    'linewidth':        2,
    'markersize':       5,              # restricted + full markers
    'markeredgewidth':  1,
    'xlim':             (0, 22),
    'ylim':             (0, 10),
    'x_major':          2,
    'x_minor':          2,
    'xlabel':           'Time post infection (days)',
    'ylabel':           'log10(PFU/mL+1)',

    # measured / data
    'meas_marker':      'D',
    'meas_markersize':  None,           # None -> markersize-1
    'meas_color':       'black',

    # restricted
    'restr_marker':     '^',
    'restr_color':      '#e09312',
    'restr_facecolor':  '#e1bf83',
    'restr_linestyle':  (0, (1, 1)),

    # full
    'full_marker':      'o',
    'full_linestyle':   'dashed',
    'full_facelighten': 0.5,

    # legend swatch colour for the full model (label colour varies per panel)
    'legend_full_color': '#1f77b4',
    'legend_ncol':       3,
}

def plot_panel_bare(fit, outpath, style=None):
    """Bare plot: measured + restricted + full. No title, legend or SSR boxes."""
    s = {**PANEL_STYLE, **(style or {})}
    key, label, opt_lag = fit['key'], fit['label'], OPT_LAG
    full_color = granger_label_color_map[label]
    dpis = fit['dpis']
    meas_ms = s['meas_markersize'] if s['meas_markersize'] is not None \
              else max(1, s['markersize'] - 1)

    fig, ax = plt.subplots(figsize=s['figsize'], dpi=s['dpi'])
    ax.plot(log_interpolated_ts_df.index.values,
            log_interpolated_ts_df['plaque_assay'].astype(float).values,
            marker=s['meas_marker'], markersize=meas_ms, color=s['meas_color'],
            linewidth=s['linewidth'])
    ax.plot(dpis[opt_lag:], fit['restr_pred'],
            marker=s['restr_marker'], markersize=s['markersize'], color=s['restr_color'],
            markeredgecolor=s['restr_color'], markerfacecolor=s['restr_facecolor'],
            markeredgewidth=s['markeredgewidth'], linewidth=s['linewidth'],
            linestyle=s['restr_linestyle'])
    ax.plot(dpis[opt_lag:], fit['full_pred'],
            marker=s['full_marker'], markersize=s['markersize'], color=full_color,
            markeredgecolor=full_color, markerfacecolor=lighten_color(full_color, s['full_facelighten']),
            markeredgewidth=s['markeredgewidth'], linewidth=s['linewidth'],
            linestyle=s['full_linestyle'])

    ax.set_xlabel(s['xlabel'])
    ax.set_ylabel(s['ylabel'])
    ax.xaxis.set_major_locator(MultipleLocator(s['x_major']))
    ax.xaxis.set_minor_locator(AutoMinorLocator(s['x_minor']))
    ax.set_xlim(*s['xlim'])
    ax.set_ylim(*s['ylim'])
    ax.set_yscale('linear')

    fig.savefig(outpath, dpi=s['dpi'], bbox_inches='tight', transparent=True)
    plt.show()
    plt.close(fig)

def save_legend(outpath, figsize=(9, 0.7), style=None):
    s = {**PANEL_STYLE, **(style or {})}
    fc = s['legend_full_color']
    handles = [
        Line2D([0], [0], color=s['meas_color'], marker=s['meas_marker'],
              linestyle='-', linewidth=s['linewidth'], label='Data'),
        Line2D([0], [0], color=s['restr_color'], marker=s['restr_marker'],
              markersize=s['markersize'], markerfacecolor=s['restr_facecolor'],
              markeredgecolor=s['restr_color'], markeredgewidth=s['markeredgewidth'],
              linewidth=s['linewidth'], linestyle=s['restr_linestyle'],
              label='Restricted model'),
        Line2D([0], [0], color=fc, marker=s['full_marker'], markersize=s['markersize'],
              markerfacecolor=lighten_color(fc, s['full_facelighten']),
              markeredgecolor=fc, markeredgewidth=s['markeredgewidth'],
              linewidth=s['linewidth'], linestyle=s['full_linestyle'],
              label='Full model'),
    ]
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.legend(handles=handles, loc='center', ncol=s['legend_ncol'], frameon=False)
    fig.savefig(outpath, dpi=s['dpi'], bbox_inches='tight', transparent=True)
    plt.show()
    plt.close(fig)

# ---- generate every element ----
elements = []
for key, etype in VALIDATED.items():
    full_win   = log_interpolated_ts_df[['plaque_assay', key]].copy()
    active_win = active_window_df(key, enrichment[key])
    is_restricted = len(active_win) < len(full_win)
    panel_style = {'figsize': FIG_SIZES.get(etype, PANEL_STYLE['figsize'])}

    series_list = [('whole', full_win)] + ([('active', active_win)] if is_restricted else [])
    for tag, win in series_list:
        fit = fit_candidate(key, win)
        fname = f'{PANEL_DIR}/{key}_{tag}.png'
        print(f"--- {key} ({etype}) | {tag} ---")
        plot_panel_bare(fit, fname, style=panel_style)
        elements.append({
            'file': fname, 'key': key, 'enrichment': etype, 'series': tag,
            'window': f"{fit['window_start']:.2f}–{fit['window_end']:.2f} dpi",
            'n': fit['n'], 'label': fit['label'],
            'p_causing': round(fit['p_causing'], 4),
            'p_caused': round(fit['p_caused'], 4),
            'SSR_restricted': round(fit['ssr_restricted'], 2),
            'SSR_full': round(fit['ssr_full'], 2),
        })

save_legend(f'{PANEL_DIR}/legend.png')

# ---- text elements for typesetting into the panel ----
panel_df = pd.DataFrame(elements)
panel_df.to_csv(f'{PANEL_DIR}/panel_values.csv', index=False)

print("\n=== Panel text elements ===")
for r in panel_df.itertuples():
    print(f"{r.key} ({r.enrichment}) | {r.series} | {r.window} | n={r.n}")
    print(f"    Label: {r.label}")
    print(f"    SSR(restricted) = {r.SSR_restricted}    SSR(full) = {r.SSR_full}")
    print(f"    p(causing) = {r.p_causing}   p(caused) = {r.p_caused}\n")

print(panel_df.to_string(index=False))

In [ ]:
val = ['PB2_217_2204', 'PB2_129_2176', 'PB2_269_2202']

# active-window BH-corrected (from the genome-wide sweep)
print("=== active window, BH-corrected ===")
print(perf_df.loc[perf_df.index.isin(val),
                  ['window_len', 'p_causing', 'p_causing_bh',
                  'granger_label_raw', 'granger_label']].to_string())

# whole-series BH-corrected (from the main analysis)
print("\n=== whole time series, BH-corrected ===")
sd = summary_df_bh[1].set_index('key')
cols = [c for c in sd.columns if 'plaque_assay' in c and ('pval' in c or 'label' in c)]
print(sd.loc[sd.index.isin(val), cols].to_string())